<a href="https://colab.research.google.com/github/mbetagonza/MentorIA-Lab/blob/main/MentorIALAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install gradio scipy scikit-image matplotlib

In [2]:
# Generar requirements.txt
!pip freeze > requirements.txt
print('requirements.txt generado.')

requirements.txt generado.


In [3]:
# Generar .gitignore
with open('.gitignore', 'w') as f:
    f.write('__pycache__/*\n')
    f.write('*.pyc\n')
    f.write('*.ipynb_checkpoints/\n')
    f.write('*.DS_Store\n')
    f.write('*.env\n')
    f.write('secrets.py\n')
    f.write('logs/\n')
    f.write('baseline_original.ipynb\n') # No queremos versionar el baseline si se trabaja en una copia
    f.write('data/\n')
    f.write('gradio_temp/\n')
    f.write('*.gradio.live\n')
print('.gitignore generado.')

.gitignore generado.


In [4]:
# =============================================================
# Simulador 2D de ultrasonido pulse-echo para Google Colab + Gradio
# Versión con 3 niveles de simulación inspirados en SIMUS
#
# Nivel 1: Reflectividad continua R(x,z) + campo de transductor por elementos
#          r_l(t) = K ∫∫ R(x,z) n(t-2z/c0) A(z) |q_l(x,z)|² dx dz
#
# Nivel 2: Reflectividad continua discretizada en puntos de grilla + RF por canal
#          + beamforming delay-and-sum.
#
# Nivel 3: Scatterers puntuales aleatorios estilo SIMUS simplificado + RF por canal
#          + beamforming delay-and-sum.
#
# En Google Colab, ejecutar primero:
# !pip -q install gradio scipy scikit-image matplotlib
# =============================================================

import numpy as np
import matplotlib.pyplot as plt
import gradio as gr
from scipy.ndimage import gaussian_filter
from scipy.signal import hilbert, fftconvolve
from skimage.draw import disk


# =============================================================
# 0. Utilidades generales
# =============================================================

def robust_norm(img, pmin=1, pmax=99):
    img = np.asarray(img, dtype=np.float64)
    lo, hi = np.percentile(img, [pmin, pmax])
    out = (img - lo) / (hi - lo + 1e-12)
    return np.clip(out, 0, 1)


def make_fig_image(img, title="", cmap="gray", extent=None, vmin=None, vmax=None):
    fig, ax = plt.subplots(figsize=(5.4, 4.2), dpi=130)
    ax.imshow(img, cmap=cmap, aspect="auto", extent=extent, origin="upper", vmin=vmin, vmax=vmax)
    ax.set_title(title)
    if extent is not None:
        ax.set_xlabel("x [mm]")
        ax.set_ylabel("z [mm]")
    ax.grid(False)
    fig.tight_layout()
    return fig


def make_line_fig(x, y, title="", xlabel="", ylabel=""):
    fig, ax = plt.subplots(figsize=(5.5, 3.5), dpi=130)
    ax.plot(x, y)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    return fig


def sincu(u):
    """sinc no normalizada: sin(u)/u."""
    return np.sinc(u / np.pi)


def interp1_uniform(y, x_float):
    """Interpolación lineal rápida sobre una señal 1D y[n] en posiciones flotantes."""
    n = len(y)
    x0 = np.floor(x_float).astype(int)
    f = x_float - x0
    valid = (x0 >= 0) & (x0 < n - 1)
    out = np.zeros_like(x_float, dtype=np.float64)
    out[valid] = (1 - f[valid]) * y[x0[valid]] + f[valid] * y[x0[valid] + 1]
    return out


# =============================================================
# 1. Phantom: c(x,z), rho(x,z), impedancia y reflectividad
# =============================================================

def generate_phantom(
    nx=256,
    nz=384,
    width_mm=40.0,
    depth_mm=70.0,
    c_bg=1540.0,
    rho_bg=1000.0,
    c_inc1=1480.0,
    rho_inc1=950.0,
    inc1_x_mm=-8.0,
    inc1_z_mm=32.0,
    inc1_r_mm=6.0,
    c_inc2=1600.0,
    rho_inc2=1060.0,
    inc2_x_mm=9.0,
    inc2_z_mm=47.0,
    inc2_r_mm=8.0,
    speckle_strength=0.025,
    seed=1,
    smooth_sigma=0.7,
):
    """Genera un phantom 2D con fondo y dos inclusiones.

    c(x,z): velocidad del sonido [m/s]
    rho(x,z): densidad [kg/m^3]
    Z(x,z)=rho*c: impedancia acústica
    R_edges(x,z)=|∇logZ|: reflectividad de interfaces
    R(x,z)=R_edges + microdispersión: reflectividad total usada por el simulador
    """
    rng = np.random.default_rng(int(seed))

    x_mm = np.linspace(-width_mm / 2, width_mm / 2, int(nx))
    z_mm = np.linspace(0, depth_mm, int(nz))
    dx_m = (x_mm[1] - x_mm[0]) * 1e-3
    dz_m = (z_mm[1] - z_mm[0]) * 1e-3

    c_map = np.full((int(nz), int(nx)), c_bg, dtype=np.float64)
    rho_map = np.full((int(nz), int(nx)), rho_bg, dtype=np.float64)

    def add_circle(c_val, rho_val, x0_mm, z0_mm, r_mm):
        ix = int(np.argmin(np.abs(x_mm - x0_mm)))
        iz = int(np.argmin(np.abs(z_mm - z0_mm)))
        rx_pix = max(1, int(r_mm / (x_mm[1] - x_mm[0])))
        rz_pix = max(1, int(r_mm / (z_mm[1] - z_mm[0])))
        r_pix = int(0.5 * (rx_pix + rz_pix))
        rr, cc = disk((iz, ix), r_pix, shape=c_map.shape)
        c_map[rr, cc] = c_val
        rho_map[rr, cc] = rho_val

    add_circle(c_inc1, rho_inc1, inc1_x_mm, inc1_z_mm, inc1_r_mm)
    add_circle(c_inc2, rho_inc2, inc2_x_mm, inc2_z_mm, inc2_r_mm)

    if smooth_sigma > 0:
        c_map = gaussian_filter(c_map, smooth_sigma)
        rho_map = gaussian_filter(rho_map, smooth_sigma)

    Z_map = rho_map * c_map
    logZ = np.log(Z_map + 1e-12)

    gz, gx = np.gradient(logZ)
    R_edges = np.sqrt(gx**2 + gz**2)
    R_edges = R_edges / (np.percentile(R_edges, 99.5) + 1e-12)
    R_edges = np.clip(R_edges, 0, 1)

    # Microdispersores: componente granular que permite speckle en los niveles 2 y 3.
    micro = rng.normal(0.0, 1.0, size=(int(nz), int(nx)))
    micro = gaussian_filter(micro, 0.35)
    micro = micro / (np.std(micro) + 1e-12)

    R_total = R_edges + speckle_strength * micro
    R_total = R_total - np.mean(R_total)

    grids = {
        "x_mm": x_mm,
        "z_mm": z_mm,
        "dx_m": dx_m,
        "dz_m": dz_m,
        "width_mm": width_mm,
        "depth_mm": depth_mm,
    }

    return {
        "c": c_map,
        "rho": rho_map,
        "Z": Z_map,
        "R_edges": R_edges,
        "R": R_total,
        "micro": micro,
        "grids": grids,
    }


# =============================================================
# 2. Transductor y pulso
# =============================================================

def gaussian_pulse(f0_mhz=5.0, cycles=2.5, dt=2e-7, duration_factor=4.0):
    """Pulso RF n(t): senoide modulada por una gaussiana."""
    f0 = f0_mhz * 1e6
    sigma_t = cycles / (2.5 * f0)
    t_max = duration_factor * sigma_t
    t = np.arange(-t_max, t_max + dt, dt)
    env = np.exp(-0.5 * (t / sigma_t) ** 2)
    pulse = env * np.cos(2 * np.pi * f0 * t)
    pulse -= np.mean(pulse)
    pulse /= np.max(np.abs(pulse)) + 1e-12
    return t, pulse


def get_element_positions(n_elements=64, pitch_mm=0.3):
    n_elements = int(n_elements)
    idx = np.arange(n_elements) - (n_elements - 1) / 2
    return idx * pitch_mm


def apodization_for_line(element_x_mm, line_x_mm, aperture_mm, kind="hann"):
    """Apodización de transmisión o recepción centrada en una línea."""
    element_x_mm = np.asarray(element_x_mm)
    half = max(aperture_mm / 2, 1e-6)
    u = (element_x_mm - line_x_mm) / half
    active = np.abs(u) <= 1
    w = np.zeros_like(element_x_mm, dtype=np.float64)

    if np.any(active):
        if kind == "rect":
            w[active] = 1.0
        else:
            # Hann centrada: vale 1 al centro y 0 en los bordes de la apertura.
            w[active] = 0.5 * (1 + np.cos(np.pi * u[active]))
    else:
        # Si la línea cae fuera de la apertura, al menos activamos el elemento más cercano.
        w[np.argmin(np.abs(element_x_mm - line_x_mm))] = 1.0

    if np.sum(w) > 0:
        w = w / (np.max(w) + 1e-12)
    return w


def compute_gaussian_beam_map(x_mm, z_mm, line_x_mm, f0_mhz=5.0, aperture_mm=16.0, focus_mm=35.0):
    """Haz gaussiano pedagógico usado como referencia rápida."""
    c0 = 1540.0
    lam_mm = c0 / (f0_mhz * 1e6) * 1e3
    f_number = max(focus_mm / max(aperture_mm, 0.5), 0.1)
    sigma_focus = max(1.2 * lam_mm * f_number, 0.25)
    divergence = lam_mm / max(aperture_mm, 0.5) * 2.2
    sigma_z = np.sqrt(sigma_focus**2 + (divergence * (z_mm - focus_mm))**2)

    X, Z = np.meshgrid(x_mm, z_mm)
    SIG = sigma_z[:, None]
    q_amp = np.exp(-0.5 * ((X - line_x_mm) / (SIG + 1e-12)) ** 2)
    axial_focus = np.exp(-0.5 * ((Z - focus_mm) / (0.85 * focus_mm + 1e-12)) ** 2)
    q_amp = q_amp * (0.35 + 0.65 * axial_focus)
    q2 = q_amp**2
    return q2 / (np.max(q2) + 1e-12)


def array_tx_field_points(
    xs_mm,
    zs_mm,
    line_x_mm,
    f0_mhz=5.0,
    n_elements=64,
    pitch_mm=0.3,
    element_width_mm=0.24,
    aperture_mm=16.0,
    focus_mm=35.0,
    c0=1540.0,
    apod_kind="hann",
    soft_baffle=True,
):
    """Campo transmitido complejo q_tx evaluado en puntos.

    Modelo 2D simplificado inspirado en SIMUS:
    q_tx(X,omega) = Σ_n W_n D(theta_n,k) exp(i k (r_n-r_focus_n)) / sqrt(r_n)

    Se usa 1/sqrt(r) porque es una aproximación acústica 2D.
    """
    xs_m = np.asarray(xs_mm, dtype=np.float64) * 1e-3
    zs_m = np.asarray(zs_mm, dtype=np.float64) * 1e-3
    xe_mm = get_element_positions(n_elements, pitch_mm)
    xe_m = xe_mm * 1e-3

    f0 = f0_mhz * 1e6
    k = 2 * np.pi * f0 / c0
    b_m = max(element_width_mm, 0.01) * 1e-3 / 2
    zf_m = max(focus_mm, 1.0) * 1e-3
    xf_m = line_x_mm * 1e-3

    w = apodization_for_line(xe_mm, line_x_mm, aperture_mm, apod_kind)

    q = np.zeros(xs_m.shape, dtype=np.complex128)
    for xe, wi in zip(xe_m, w):
        if wi == 0:
            continue
        r = np.sqrt((xs_m - xe)**2 + zs_m**2) + 1e-12
        r_focus = np.sqrt((xf_m - xe)**2 + zf_m**2) + 1e-12
        sin_theta = (xs_m - xe) / r
        cos_theta = np.maximum(zs_m / r, 0)
        D = sincu(k * b_m * sin_theta)
        if soft_baffle:
            D = D * cos_theta
        q += wi * D * np.exp(1j * k * (r - r_focus)) / np.sqrt(r)

    # Normalización de amplitud para evitar que cambie drásticamente con N.
    amp = np.abs(q)
    q = q / (np.percentile(amp, 99.5) + 1e-12)
    return q


def compute_array_beam_map(
    x_mm,
    z_mm,
    line_x_mm,
    f0_mhz=5.0,
    n_elements=64,
    pitch_mm=0.3,
    element_width_mm=0.24,
    aperture_mm=16.0,
    focus_mm=35.0,
    c0=1540.0,
):
    X, Z = np.meshgrid(x_mm, z_mm)
    q = array_tx_field_points(
        X.ravel(), Z.ravel(), line_x_mm,
        f0_mhz=f0_mhz,
        n_elements=n_elements,
        pitch_mm=pitch_mm,
        element_width_mm=element_width_mm,
        aperture_mm=aperture_mm,
        focus_mm=focus_mm,
        c0=c0,
    )
    q2 = np.abs(q.reshape(len(z_mm), len(x_mm)))**2
    return q2 / (np.max(q2) + 1e-12)


def rx_element_sensitivity_points(xs_mm, zs_mm, element_x_mm, f0_mhz=5.0, element_width_mm=0.24, c0=1540.0, soft_baffle=True):
    xs_m = np.asarray(xs_mm) * 1e-3
    zs_m = np.asarray(zs_mm) * 1e-3
    xe_m = element_x_mm * 1e-3
    f0 = f0_mhz * 1e6
    k = 2 * np.pi * f0 / c0
    b_m = max(element_width_mm, 0.01) * 1e-3 / 2
    r = np.sqrt((xs_m - xe_m)**2 + zs_m**2) + 1e-12
    sin_theta = (xs_m - xe_m) / r
    cos_theta = np.maximum(zs_m / r, 0)
    D = sincu(k * b_m * sin_theta)
    if soft_baffle:
        D = D * cos_theta
    return D / np.sqrt(r)


# =============================================================
# 3. Atenuación
# =============================================================

def attenuation_profile(z_mm, mu_db_cm_mhz=0.5, f0_mhz=5.0):
    """Atenuación round-trip en amplitud.

    alpha_db/cm = mu_db_cm_mhz * f0_mhz
    factor = 10^[-(2 alpha z_cm)/20]
    """
    z_cm = np.asarray(z_mm) / 10.0
    alpha_db_cm = mu_db_cm_mhz * f0_mhz
    return 10 ** (-(2.0 * alpha_db_cm * z_cm) / 20.0)


# =============================================================
# 4A. Nivel 1: reflectividad continua + q por elementos
# =============================================================

def simulate_level1_continuous(
    phantom,
    n_lines=96,
    selected_line=48,
    f0_mhz=5.0,
    cycles=2.5,
    n_elements=64,
    pitch_mm=0.3,
    element_width_mm=0.24,
    aperture_mm=16.0,
    focus_mm=35.0,
    mu_db_cm_mhz=0.5,
    K=1.0,
    dynamic_range_db=55.0,
    beam_kind="Elementos por difracción",
):
    R = phantom["R"]
    grids = phantom["grids"]
    x_mm = grids["x_mm"]
    z_mm = grids["z_mm"]
    dx_m = grids["dx_m"]
    dz_m = grids["dz_m"]
    nz, nx = R.shape

    c0 = float(np.mean(phantom["c"]))
    dt_depth = 2.0 * dz_m / c0
    _, pulse = gaussian_pulse(f0_mhz=f0_mhz, cycles=cycles, dt=dt_depth)

    line_positions = np.linspace(x_mm.min(), x_mm.max(), int(n_lines))
    RF = np.zeros((nz, int(n_lines)), dtype=np.float64)
    pre_echo = np.zeros_like(RF)
    atten = attenuation_profile(z_mm, mu_db_cm_mhz=mu_db_cm_mhz, f0_mhz=f0_mhz)

    for li, x0 in enumerate(line_positions):
        if beam_kind.startswith("Gaussiano"):
            q2 = compute_gaussian_beam_map(x_mm, z_mm, x0, f0_mhz, aperture_mm, focus_mm)
        else:
            q2 = compute_array_beam_map(
                x_mm, z_mm, x0,
                f0_mhz=f0_mhz,
                n_elements=n_elements,
                pitch_mm=pitch_mm,
                element_width_mm=element_width_mm,
                aperture_mm=aperture_mm,
                focus_mm=focus_mm,
                c0=c0,
            )

        A_z = K * np.sum(R * q2, axis=1) * dx_m
        A_z = A_z * atten
        pre_echo[:, li] = A_z
        RF[:, li] = fftconvolve(A_z, pulse, mode="same") * dz_m

    ENV = np.abs(hilbert(RF, axis=0))
    ENV /= np.max(ENV) + 1e-12
    B_db = 20.0 * np.log10(ENV + 1e-6)
    B_norm = np.clip((B_db + dynamic_range_db) / dynamic_range_db, 0, 1)

    selected_line = int(np.clip(selected_line, 0, int(n_lines) - 1))
    if beam_kind.startswith("Gaussiano"):
        q2_sel = compute_gaussian_beam_map(x_mm, z_mm, line_positions[selected_line], f0_mhz, aperture_mm, focus_mm)
    else:
        q2_sel = compute_array_beam_map(
            x_mm, z_mm, line_positions[selected_line],
            f0_mhz=f0_mhz,
            n_elements=n_elements,
            pitch_mm=pitch_mm,
            element_width_mm=element_width_mm,
            aperture_mm=aperture_mm,
            focus_mm=focus_mm,
            c0=c0,
        )

    return {
        "mode": "Nivel 1: continuo + q por elementos",
        "RF": RF,
        "ENV": ENV,
        "B": B_norm,
        "B_db": B_db,
        "A_mode": ENV[:, selected_line],
        "RF_line": RF[:, selected_line],
        "pre_line": pre_echo[:, selected_line],
        "q2_selected": q2_sel,
        "line_positions": line_positions,
        "z_mm": z_mm,
        "x_mm": x_mm,
        "selected_line": selected_line,
        "dt": dt_depth,
        "n_points": int(nx * nz),
    }


# =============================================================
# 4B. Puntos discretos: grilla regular o scatterers aleatorios
# =============================================================

def regular_grid_points_from_reflectivity(phantom, stride=3, max_points=8000):
    """Convierte R(x,z) en puntos regulares de grilla.

    Esto es parecido a tratar cada celda como un scatterer fijo en el centro del pixel.
    """
    R = phantom["R"]
    grids = phantom["grids"]
    x_mm = grids["x_mm"]
    z_mm = grids["z_mm"]
    dx_m = grids["dx_m"]
    dz_m = grids["dz_m"]

    zz_idx = np.arange(0, R.shape[0], int(stride))
    xx_idx = np.arange(0, R.shape[1], int(stride))
    ZZ, XX = np.meshgrid(zz_idx, xx_idx, indexing="ij")
    xs = x_mm[XX.ravel()]
    zs = z_mm[ZZ.ravel()]
    amps = R[ZZ.ravel(), XX.ravel()] * dx_m * dz_m * (stride**2)

    # Quitar puntos casi nulos para acelerar.
    keep = np.abs(amps) > np.percentile(np.abs(amps), 40)
    xs, zs, amps = xs[keep], zs[keep], amps[keep]

    if len(xs) > max_points:
        rng = np.random.default_rng(0)
        prob = np.abs(amps) + 1e-12
        prob = prob / np.sum(prob)
        idx = rng.choice(len(xs), size=int(max_points), replace=False, p=prob)
        xs, zs, amps = xs[idx], zs[idx], amps[idx]

    return xs, zs, amps


def scatterers_from_phantom(phantom, n_scatterers=6000, seed=1, interface_gain=1.5, speckle_gain=0.6):
    """Genera scatterers aleatorios a partir del phantom.

    La amplitud combina:
    - componente aleatoria de speckle
    - componente de interfaces desde R_edges
    """
    rng = np.random.default_rng(int(seed))
    grids = phantom["grids"]
    x_mm_grid = grids["x_mm"]
    z_mm_grid = grids["z_mm"]
    R_edges = phantom["R_edges"]
    micro = phantom["micro"]

    xs = rng.uniform(x_mm_grid.min(), x_mm_grid.max(), int(n_scatterers))
    zs = rng.uniform(max(0.2, z_mm_grid.min()), z_mm_grid.max(), int(n_scatterers))

    # Interpolación nearest para mantenerlo simple y rápido.
    ix = np.clip(np.searchsorted(x_mm_grid, xs), 1, len(x_mm_grid)-1)
    iz = np.clip(np.searchsorted(z_mm_grid, zs), 1, len(z_mm_grid)-1)
    ix = np.where(np.abs(x_mm_grid[ix] - xs) < np.abs(x_mm_grid[ix-1] - xs), ix, ix-1)
    iz = np.where(np.abs(z_mm_grid[iz] - zs) < np.abs(z_mm_grid[iz-1] - zs), iz, iz-1)

    interface_amp = R_edges[iz, ix]
    tissue_amp = np.abs(micro[iz, ix])

    # Coeficientes reales con signo aleatorio. La fase RF se generará por los retardos del pulso.
    signs = rng.choice([-1.0, 1.0], size=int(n_scatterers))
    amps = signs * (speckle_gain * (0.25 + tissue_amp) + interface_gain * interface_amp)
    amps = amps / (np.std(amps) + 1e-12)
    amps = amps * grids["dx_m"] * grids["dz_m"]
    return xs, zs, amps


# =============================================================
# 4C. Nivel 2/3: RF por canal + DAS
# =============================================================

def simulate_rf_channels_for_line(
    xs_mm,
    zs_mm,
    amps,
    phantom,
    line_x_mm,
    f0_mhz=5.0,
    cycles=2.5,
    n_elements=32,
    pitch_mm=0.3,
    element_width_mm=0.24,
    aperture_mm=12.0,
    focus_mm=35.0,
    mu_db_cm_mhz=0.5,
    K=1.0,
    nt=None,
):
    """Genera RF por canal para una línea.

    Simplificación geométrica:
    - transmisión efectiva tipo línea vertical: tau_tx = z/c0
    - recepción por elemento: tau_rx_m = sqrt((x-x_m)^2+z^2)/c0
    - el campo transmitido q_tx sí se calcula por elementos para ponderar amplitud.
    """
    grids = phantom["grids"]
    z_mm_grid = grids["z_mm"]
    dz_m = grids["dz_m"]
    c0 = float(np.mean(phantom["c"]))
    dt = 2.0 * dz_m / c0
    if nt is None:
        nt = len(z_mm_grid)

    _, pulse = gaussian_pulse(f0_mhz=f0_mhz, cycles=cycles, dt=dt)
    element_x_mm = get_element_positions(n_elements, pitch_mm)
    rx_w = apodization_for_line(element_x_mm, line_x_mm, aperture_mm, kind="hann")

    # Campo transmitido en los puntos.
    qtx = array_tx_field_points(
        xs_mm, zs_mm, line_x_mm,
        f0_mhz=f0_mhz,
        n_elements=n_elements,
        pitch_mm=pitch_mm,
        element_width_mm=element_width_mm,
        aperture_mm=aperture_mm,
        focus_mm=focus_mm,
        c0=c0,
    )
    qtx_amp = np.abs(qtx)
    qtx_amp = qtx_amp / (np.percentile(qtx_amp, 99.5) + 1e-12)

    atten = attenuation_profile(zs_mm, mu_db_cm_mhz=mu_db_cm_mhz, f0_mhz=f0_mhz)

    xs_m = xs_mm * 1e-3
    zs_m = zs_mm * 1e-3
    RF_ch = np.zeros((int(nt), int(n_elements)), dtype=np.float64)

    tau_tx = zs_m / c0

    for mi, xe_mm in enumerate(element_x_mm):
        xe_m = xe_mm * 1e-3
        r_rx = np.sqrt((xs_m - xe_m)**2 + zs_m**2)
        tau = tau_tx + r_rx / c0
        idx = tau / dt
        i0 = np.floor(idx).astype(int)
        frac = idx - i0
        valid = (i0 >= 0) & (i0 < int(nt) - 1)

        qrx = rx_element_sensitivity_points(xs_mm, zs_mm, xe_mm, f0_mhz, element_width_mm, c0)
        qrx_amp = np.abs(qrx)
        qrx_amp = qrx_amp / (np.percentile(qrx_amp, 99.5) + 1e-12)

        impulse = np.zeros(int(nt), dtype=np.float64)
        val = K * amps * qtx_amp * qrx_amp * atten
        np.add.at(impulse, i0[valid], val[valid] * (1 - frac[valid]))
        np.add.at(impulse, i0[valid] + 1, val[valid] * frac[valid])

        RF_ch[:, mi] = fftconvolve(impulse, pulse, mode="same")

    return RF_ch, dt, element_x_mm


def das_beamform_line(RF_ch, z_mm, line_x_mm, element_x_mm, aperture_mm=12.0, c0=1540.0, dt=1e-7):
    """Delay-and-sum de recepción para una línea."""
    z_m = z_mm * 1e-3
    x0_m = line_x_mm * 1e-3
    rx_w = apodization_for_line(element_x_mm, line_x_mm, aperture_mm, kind="hann")

    b = np.zeros_like(z_mm, dtype=np.float64)
    for mi, xe_mm in enumerate(element_x_mm):
        xe_m = xe_mm * 1e-3
        tau = z_m / c0 + np.sqrt((x0_m - xe_m)**2 + z_m**2) / c0
        samples = tau / dt
        b += rx_w[mi] * interp1_uniform(RF_ch[:, mi], samples)

    b /= (np.sum(rx_w) + 1e-12)
    return b


def simulate_channel_das(
    phantom,
    level="Nivel 2: grilla continua discretizada + RF por canal + DAS",
    n_lines=64,
    selected_line=32,
    f0_mhz=5.0,
    cycles=2.5,
    n_elements=32,
    pitch_mm=0.3,
    element_width_mm=0.24,
    aperture_mm=12.0,
    focus_mm=35.0,
    mu_db_cm_mhz=0.5,
    K=1.0,
    dynamic_range_db=55.0,
    grid_stride=3,
    n_scatterers=5000,
    scatter_seed=1,
    interface_gain=1.5,
    speckle_gain=0.6,
):
    grids = phantom["grids"]
    x_mm = grids["x_mm"]
    z_mm = grids["z_mm"]
    c0 = float(np.mean(phantom["c"]))
    nt = len(z_mm)

    if level.startswith("Nivel 3"):
        xs, zs, amps = scatterers_from_phantom(
            phantom,
            n_scatterers=int(n_scatterers),
            seed=int(scatter_seed),
            interface_gain=interface_gain,
            speckle_gain=speckle_gain,
        )
        mode_name = "Nivel 3: scatterers puntuales + RF por canal + DAS"
    else:
        xs, zs, amps = regular_grid_points_from_reflectivity(
            phantom,
            stride=int(grid_stride),
            max_points=int(n_scatterers),
        )
        mode_name = "Nivel 2: R(x,z) discretizada + RF por canal + DAS"

    line_positions = np.linspace(x_mm.min(), x_mm.max(), int(n_lines))
    RF_bf = np.zeros((nt, int(n_lines)), dtype=np.float64)
    selected_line = int(np.clip(selected_line, 0, int(n_lines) - 1))
    RF_ch_selected = None
    dt_selected = None
    element_x_selected = None

    for li, x0 in enumerate(line_positions):
        RF_ch, dt, element_x_mm = simulate_rf_channels_for_line(
            xs, zs, amps, phantom, x0,
            f0_mhz=f0_mhz,
            cycles=cycles,
            n_elements=n_elements,
            pitch_mm=pitch_mm,
            element_width_mm=element_width_mm,
            aperture_mm=aperture_mm,
            focus_mm=focus_mm,
            mu_db_cm_mhz=mu_db_cm_mhz,
            K=K,
            nt=nt,
        )
        RF_bf[:, li] = das_beamform_line(
            RF_ch, z_mm, x0, element_x_mm,
            aperture_mm=aperture_mm,
            c0=c0,
            dt=dt,
        )
        if li == selected_line:
            RF_ch_selected = RF_ch
            dt_selected = dt
            element_x_selected = element_x_mm

    ENV = np.abs(hilbert(RF_bf, axis=0))
    ENV /= np.max(ENV) + 1e-12
    B_db = 20.0 * np.log10(ENV + 1e-6)
    B_norm = np.clip((B_db + dynamic_range_db) / dynamic_range_db, 0, 1)

    q2_sel = compute_array_beam_map(
        x_mm, z_mm, line_positions[selected_line],
        f0_mhz=f0_mhz,
        n_elements=n_elements,
        pitch_mm=pitch_mm,
        element_width_mm=element_width_mm,
        aperture_mm=aperture_mm,
        focus_mm=focus_mm,
        c0=c0,
    )

    return {
        "mode": mode_name,
        "RF": RF_bf,
        "ENV": ENV,
        "B": B_norm,
        "B_db": B_db,
        "A_mode": ENV[:, selected_line],
        "RF_line": RF_bf[:, selected_line],
        "RF_channels_selected": RF_ch_selected,
        "q2_selected": q2_sel,
        "line_positions": line_positions,
        "z_mm": z_mm,
        "x_mm": x_mm,
        "selected_line": selected_line,
        "dt": dt_selected,
        "element_x_mm": element_x_selected,
        "scatter_x_mm": xs,
        "scatter_z_mm": zs,
        "scatter_amp": amps,
        "n_points": len(xs),
    }


# =============================================================
# 5. Callbacks Gradio
# =============================================================

def cb_generate_phantom(
    nx, nz, width_mm, depth_mm,
    c_bg, rho_bg,
    c_inc1, rho_inc1, inc1_x, inc1_z, inc1_r,
    c_inc2, rho_inc2, inc2_x, inc2_z, inc2_r,
    speckle_strength, seed, smooth_sigma,
):
    phantom = generate_phantom(
        int(nx), int(nz), width_mm, depth_mm,
        c_bg, rho_bg,
        c_inc1, rho_inc1, inc1_x, inc1_z, inc1_r,
        c_inc2, rho_inc2, inc2_x, inc2_z, inc2_r,
        speckle_strength, int(seed), smooth_sigma,
    )
    grids = phantom["grids"]
    extent = [grids["x_mm"].min(), grids["x_mm"].max(), grids["z_mm"].max(), grids["z_mm"].min()]
    fig_c = make_fig_image(phantom["c"], "Velocidad c(x,z) [m/s]", cmap="viridis", extent=extent)
    fig_rho = make_fig_image(phantom["rho"], "Densidad ρ(x,z) [kg/m³]", cmap="magma", extent=extent)
    fig_R = make_fig_image(robust_norm(np.abs(phantom["R"])), "Reflectividad |R(x,z)|", cmap="gray", extent=extent)
    return phantom, fig_c, fig_rho, fig_R


def cb_preview_transducer(phantom, n_lines, selected_line, f0_mhz, cycles, n_elements, pitch_mm, element_width_mm, aperture_mm, focus_mm, beam_kind):
    if phantom is None:
        phantom = generate_phantom()
    grids = phantom["grids"]
    x_mm = grids["x_mm"]
    z_mm = grids["z_mm"]
    c0 = float(np.mean(phantom["c"]))
    line_positions = np.linspace(x_mm.min(), x_mm.max(), int(n_lines))
    selected_line = int(np.clip(selected_line, 0, int(n_lines)-1))
    x0 = line_positions[selected_line]

    if beam_kind.startswith("Gaussiano"):
        q2 = compute_gaussian_beam_map(x_mm, z_mm, x0, f0_mhz, aperture_mm, focus_mm)
    else:
        q2 = compute_array_beam_map(
            x_mm, z_mm, x0,
            f0_mhz=f0_mhz,
            n_elements=int(n_elements),
            pitch_mm=pitch_mm,
            element_width_mm=element_width_mm,
            aperture_mm=aperture_mm,
            focus_mm=focus_mm,
            c0=c0,
        )

    extent = [x_mm.min(), x_mm.max(), z_mm.max(), z_mm.min()]
    fig_q = make_fig_image(q2, f"Two-way beam |q_l(x,z)|², línea {selected_line}", cmap="inferno", extent=extent)

    dt = 2 * grids["dz_m"] / c0
    t, pulse = gaussian_pulse(f0_mhz, cycles, dt)
    fig_p = make_line_fig(t * 1e6, pulse, "Pulso RF n(t)", "t [µs]", "amplitud")
    return fig_q, fig_p


def cb_preview_attenuation(phantom, mu_db_cm_mhz, f0_mhz, K):
    if phantom is None:
        phantom = generate_phantom()
    z_mm = phantom["grids"]["z_mm"]
    att = attenuation_profile(z_mm, mu_db_cm_mhz, f0_mhz)
    fig = make_line_fig(z_mm, K * att, "Factor K · atenuación round-trip", "z [mm]", "amplitud relativa")
    return fig


def cb_run_simulation(
    phantom,
    sim_level,
    beam_kind,
    n_lines,
    selected_line,
    f0_mhz,
    cycles,
    n_elements,
    pitch_mm,
    element_width_mm,
    aperture_mm,
    focus_mm,
    mu_db_cm_mhz,
    K,
    dynamic_range_db,
    grid_stride,
    n_scatterers,
    scatter_seed,
    interface_gain,
    speckle_gain,
):
    if phantom is None:
        phantom = generate_phantom()

    if sim_level.startswith("Nivel 1"):
        sim = simulate_level1_continuous(
            phantom=phantom,
            n_lines=int(n_lines),
            selected_line=int(selected_line),
            f0_mhz=f0_mhz,
            cycles=cycles,
            n_elements=int(n_elements),
            pitch_mm=pitch_mm,
            element_width_mm=element_width_mm,
            aperture_mm=aperture_mm,
            focus_mm=focus_mm,
            mu_db_cm_mhz=mu_db_cm_mhz,
            K=K,
            dynamic_range_db=dynamic_range_db,
            beam_kind=beam_kind,
        )
    else:
        sim = simulate_channel_das(
            phantom=phantom,
            level=sim_level,
            n_lines=int(n_lines),
            selected_line=int(selected_line),
            f0_mhz=f0_mhz,
            cycles=cycles,
            n_elements=int(n_elements),
            pitch_mm=pitch_mm,
            element_width_mm=element_width_mm,
            aperture_mm=aperture_mm,
            focus_mm=focus_mm,
            mu_db_cm_mhz=mu_db_cm_mhz,
            K=K,
            dynamic_range_db=dynamic_range_db,
            grid_stride=int(grid_stride),
            n_scatterers=int(n_scatterers),
            scatter_seed=int(scatter_seed),
            interface_gain=interface_gain,
            speckle_gain=speckle_gain,
        )

    x_lines = sim["line_positions"]
    z_mm = sim["z_mm"]
    extent_b = [x_lines.min(), x_lines.max(), z_mm.max(), z_mm.min()]

    fig_b = make_fig_image(sim["B"], f"Imagen B-mode simulada — {sim['mode']}", cmap="gray", extent=extent_b, vmin=0, vmax=1)
    fig_rf = make_fig_image(robust_norm(sim["RF"]), "RF beamformed por líneas", cmap="gray", extent=extent_b)
    fig_a = make_line_fig(z_mm, sim["A_mode"], f"A-mode / envolvente, línea {sim['selected_line']}", "z [mm]", "amplitud")
    fig_raw = make_line_fig(z_mm, sim["RF_line"], f"RF línea beamformed {sim['selected_line']}", "z [mm]", "amplitud")

    # Figura de canales si existe nivel 2/3.
    if sim.get("RF_channels_selected") is not None:
        ch = sim["RF_channels_selected"]
        extent_ch = [0, ch.shape[1]-1, z_mm.max(), z_mm.min()]
        fig_ch = make_fig_image(robust_norm(ch), "RF por canal antes de DAS, línea seleccionada", cmap="gray", extent=extent_ch)
    else:
        fig_ch = make_fig_image(np.zeros((16, 16)), "Nivel 1 no genera RF por canal", cmap="gray")

    # Figura de puntos/scatterers si existe.
    if sim.get("scatter_x_mm") is not None:
        fig_sc, ax = plt.subplots(figsize=(5.4, 4.2), dpi=130)
        ax.scatter(sim["scatter_x_mm"], sim["scatter_z_mm"], s=1, c=np.sign(sim["scatter_amp"]), cmap="coolwarm", alpha=0.35)
        ax.set_title(f"Puntos usados en la simulación: {sim['n_points']}")
        ax.set_xlabel("x [mm]")
        ax.set_ylabel("z [mm]")
        ax.set_ylim(z_mm.max(), z_mm.min())
        ax.set_xlim(sim["x_mm"].min(), sim["x_mm"].max())
        fig_sc.tight_layout()
    else:
        fig_sc = make_fig_image(np.zeros((16, 16)), f"Nivel 1: integración continua en grilla ({sim['n_points']} celdas)", cmap="gray")

    info = (
        f"Modo: {sim['mode']}\n"
        f"Líneas: {int(n_lines)}\n"
        f"Elementos: {int(n_elements)}\n"
        f"Puntos/celdas usados: {sim['n_points']}\n"
        f"Frecuencia: {f0_mhz:.2f} MHz\n"
        f"Foco: {focus_mm:.1f} mm\n"
        f"Apertura efectiva: {aperture_mm:.1f} mm\n"
    )

    return sim, fig_b, fig_rf, fig_a, fig_raw, fig_ch, fig_sc, info


# =============================================================
# 6. Interfaz Gradio
# =============================================================

def build_app():
    with gr.Blocks(title="Simulador 2D de Ultrasonido Pulse-Echo") as demo:
        gr.Markdown(
            r"""
            # Simulador 2D de ultrasonido pulse-echo

            Esta versión incorpora tres niveles de simulación:

            **Nivel 1:** reflectividad continua + campo del transductor por elementos.
            **Nivel 2:** reflectividad continua discretizada + RF por canal + delay-and-sum.
            **Nivel 3:** scatterers puntuales estilo SIMUS simplificado + RF por canal + delay-and-sum.

            La ecuación base del nivel continuo es:

            \[
r_l(t)=K\int\int R(x,z)\,n\left(t-\frac{2z}{c_0}\right)\,e^{-2\mu_a z}\,|q_l(x,z)|^2\,dx\,dz
            \]
            """
        )

        phantom_state = gr.State(None)
        sim_state = gr.State(None)

        with gr.Tab("1. Phantom: c, ρ, Z y R"):
            gr.Markdown(r"Diseño simple de phantom 2D. La reflectividad se aproxima desde cambios locales de impedancia acústica, \(Z=\rho c\), más una microdispersión controlable.")
            with gr.Row():
                with gr.Column(scale=1):
                    nx = gr.Slider(96, 512, value=256, step=16, label="Nx")
                    nz = gr.Slider(128, 640, value=384, step=16, label="Nz")
                    width_mm = gr.Slider(20, 80, value=40, step=1, label="Ancho lateral [mm]")
                    depth_mm = gr.Slider(30, 120, value=70, step=1, label="Profundidad [mm]")
                    c_bg = gr.Slider(1400, 1650, value=1540, step=5, label="c fondo [m/s]")
                    rho_bg = gr.Slider(850, 1150, value=1000, step=5, label="ρ fondo [kg/m³]")
                with gr.Column(scale=1):
                    gr.Markdown("### Inclusión 1")
                    c_inc1 = gr.Slider(1400, 1700, value=1480, step=5, label="c inc. 1 [m/s]")
                    rho_inc1 = gr.Slider(850, 1200, value=950, step=5, label="ρ inc. 1 [kg/m³]")
                    inc1_x = gr.Slider(-30, 30, value=-8, step=0.5, label="x inc. 1 [mm]")
                    inc1_z = gr.Slider(5, 110, value=32, step=0.5, label="z inc. 1 [mm]")
                    inc1_r = gr.Slider(1, 20, value=6, step=0.5, label="radio inc. 1 [mm]")
                with gr.Column(scale=1):
                    gr.Markdown("### Inclusión 2")
                    c_inc2 = gr.Slider(1400, 1700, value=1600, step=5, label="c inc. 2 [m/s]")
                    rho_inc2 = gr.Slider(850, 1200, value=1060, step=5, label="ρ inc. 2 [kg/m³]")
                    inc2_x = gr.Slider(-30, 30, value=9, step=0.5, label="x inc. 2 [mm]")
                    inc2_z = gr.Slider(5, 110, value=47, step=0.5, label="z inc. 2 [mm]")
                    inc2_r = gr.Slider(1, 20, value=8, step=0.5, label="radio inc. 2 [mm]")
                    speckle_strength = gr.Slider(0, 0.15, value=0.025, step=0.005, label="Microdispersión para speckle")
                    seed = gr.Number(value=1, precision=0, label="Seed")
                    smooth_sigma = gr.Slider(0, 3, value=0.7, step=0.1, label="Suavizado de interfaces [pix]")

            btn_phantom = gr.Button("Generar phantom", variant="primary")
            with gr.Row():
                fig_c = gr.Plot(label="c(x,z)")
                fig_rho = gr.Plot(label="ρ(x,z)")
                fig_R = gr.Plot(label="R(x,z)")

        with gr.Tab("2. Transductor y pulso RF"):
            gr.Markdown(r"Define geometría del transductor, campo espacial \(q_l\) y pulso temporal \(n(t)\).")
            with gr.Row():
                with gr.Column():
                    sim_level = gr.Dropdown(
                        choices=[
                            "Nivel 1: continuo + campo por elementos",
                            "Nivel 2: grilla continua discretizada + RF por canal + DAS",
                            "Nivel 3: scatterers puntuales + RF por canal + DAS",
                        ],
                        value="Nivel 1: continuo + campo por elementos",
                        label="Nivel de simulación",
                    )
                    beam_kind = gr.Dropdown(
                        choices=["Elementos por difracción", "Gaussiano pedagógico"],
                        value="Elementos por difracción",
                        label="Modelo de haz para Nivel 1 / preview",
                    )
                    n_lines = gr.Slider(16, 160, value=64, step=4, label="Número de líneas B-mode")
                    selected_line = gr.Slider(0, 159, value=32, step=1, label="Línea seleccionada")
                    f0_mhz = gr.Slider(1, 15, value=5.0, step=0.25, label="Frecuencia central f0 [MHz]")
                    cycles = gr.Slider(1, 8, value=2.5, step=0.25, label="Número de ciclos del pulso")
                with gr.Column():
                    n_elements = gr.Slider(8, 128, value=32, step=4, label="Número de elementos")
                    pitch_mm = gr.Slider(0.1, 1.0, value=0.30, step=0.01, label="Pitch [mm]")
                    element_width_mm = gr.Slider(0.05, 0.9, value=0.24, step=0.01, label="Ancho de elemento [mm]")
                    aperture_mm = gr.Slider(2, 40, value=12, step=0.5, label="Apertura efectiva [mm]")
                    focus_mm = gr.Slider(5, 100, value=35, step=1, label="Foco [mm]")
                    btn_tx = gr.Button("Previsualizar transductor/pulso")
                with gr.Column():
                    fig_q = gr.Plot(label="Patrón espacial |q|²")
                    fig_pulse = gr.Plot(label="Pulso RF")

        with gr.Tab("3. Atenuación, K y scatterers"):
            gr.Markdown("La atenuación se modela en amplitud como factor round-trip. En los niveles 2 y 3 se generan puntos discretos para simular RF por canal.")
            with gr.Row():
                with gr.Column():
                    mu_db_cm_mhz = gr.Slider(0, 2.0, value=0.5, step=0.05, label="μ [dB/(cm·MHz)]")
                    K = gr.Slider(0.1, 50, value=1.0, step=0.1, label="Factor K")
                    dynamic_range_db = gr.Slider(20, 90, value=55, step=1, label="Rango dinámico B-mode [dB]")
                    btn_att = gr.Button("Previsualizar atenuación")
                with gr.Column():
                    grid_stride = gr.Slider(1, 8, value=3, step=1, label="Nivel 2: stride de grilla")
                    n_scatterers = gr.Slider(500, 20000, value=5000, step=500, label="Nivel 2/3: máximo de puntos/scatterers")
                    scatter_seed = gr.Number(value=2, precision=0, label="Nivel 3: seed scatterers")
                    interface_gain = gr.Slider(0, 5, value=1.5, step=0.1, label="Nivel 3: ganancia de interfaces")
                    speckle_gain = gr.Slider(0, 3, value=0.6, step=0.05, label="Nivel 3: ganancia de speckle")
                with gr.Column():
                    fig_att = gr.Plot(label="K · atenuación")

        with gr.Tab("4. Resultado: RF, A-mode, B-mode y canales"):
            gr.Markdown("Ejecuta la simulación seleccionada. Los niveles 2 y 3 pueden tardar más porque calculan RF por canal y DAS.")
            btn_sim = gr.Button("Simular ultrasonido", variant="primary")
            sim_info = gr.Textbox(label="Resumen de simulación", lines=8)
            with gr.Row():
                fig_b = gr.Plot(label="B-mode")
                fig_rf = gr.Plot(label="RF beamformed")
            with gr.Row():
                fig_a = gr.Plot(label="A-mode")
                fig_raw = gr.Plot(label="RF línea seleccionada")
            with gr.Row():
                fig_ch = gr.Plot(label="RF por canal")
                fig_sc = gr.Plot(label="Puntos/scatterers")

        # Eventos
        btn_phantom.click(
            cb_generate_phantom,
            inputs=[
                nx, nz, width_mm, depth_mm,
                c_bg, rho_bg,
                c_inc1, rho_inc1, inc1_x, inc1_z, inc1_r,
                c_inc2, rho_inc2, inc2_x, inc2_z, inc2_r,
                speckle_strength, seed, smooth_sigma,
            ],
            outputs=[phantom_state, fig_c, fig_rho, fig_R],
        )

        btn_tx.click(
            cb_preview_transducer,
            inputs=[phantom_state, n_lines, selected_line, f0_mhz, cycles, n_elements, pitch_mm, element_width_mm, aperture_mm, focus_mm, beam_kind],
            outputs=[fig_q, fig_pulse],
        )

        btn_att.click(
            cb_preview_attenuation,
            inputs=[phantom_state, mu_db_cm_mhz, f0_mhz, K],
            outputs=[fig_att],
        )

        btn_sim.click(
            cb_run_simulation,
            inputs=[
                phantom_state,
                sim_level,
                beam_kind,
                n_lines,
                selected_line,
                f0_mhz,
                cycles,
                n_elements,
                pitch_mm,
                element_width_mm,
                aperture_mm,
                focus_mm,
                mu_db_cm_mhz,
                K,
                dynamic_range_db,
                grid_stride,
                n_scatterers,
                scatter_seed,
                interface_gain,
                speckle_gain,
            ],
            outputs=[sim_state, fig_b, fig_rf, fig_a, fig_raw, fig_ch, fig_sc, sim_info],
        )

        demo.load(
            cb_generate_phantom,
            inputs=[
                nx, nz, width_mm, depth_mm,
                c_bg, rho_bg,
                c_inc1, rho_inc1, inc1_x, inc1_z, inc1_r,
                c_inc2, rho_inc2, inc2_x, inc2_z, inc2_r,
                speckle_strength, seed, smooth_sigma,
            ],
            outputs=[phantom_state, fig_c, fig_rho, fig_R],
        )

    return demo


# =============================================================
# 7. Ejecución
# =============================================================
# En Colab:
# demo = build_app()
# demo.launch(share=True, debug=True)

if __name__ == "__main__":
    demo = build_app()
    demo.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4cce8926adc348f9d8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://4cce8926adc348f9d8.gradio.live


In [5]:
import numpy as np

def extraer_metricas_baseline_phantom(seed=1):
    # Invoca la función generate_phantom() definida en el esqueleto original
    phantom = generate_phantom(seed=seed)

    print(f"=== MÉTRICAS DE REFERENCIA BASELINE PHANTOM (Seed = {seed}) ===")
    print("----------------------------------------------------------------")

    metrics = {}
    # Claves reales que devuelve la función generate_phantom()
    keys_to_test = ["c", "rho", "Z", "R"]

    for key in keys_to_test:
        arr = phantom[key]
        metrics[key] = {
            "shape": arr.shape,
            "min": float(np.min(arr)),
            "max": float(np.max(arr)),
            "mean": float(np.mean(arr)),
            "checksum_abs": float(np.sum(np.abs(arr)))
        }
        print(f"Clave [{key}]:")
        print(f"  - Dimensiones (shape) : {metrics[key]['shape']}")
        print(f"  - Rango [Mín, Máx]    : [{metrics[key]['min']:.4f}, {metrics[key]['max']:.4f}]")
        print(f"  - Promedio (mean)     : {metrics[key]['mean']:.6f}")
        print(f"  - Checksum (Suma Abs) : {metrics[key]['checksum_abs']:.6f}")
        print("----------------------------------------------------------------")

    return metrics

# Ejecutar extracción
baseline_phantom_metrics = extraer_metricas_baseline_phantom(seed=1)

=== MÉTRICAS DE REFERENCIA BASELINE PHANTOM (Seed = 1) ===
----------------------------------------------------------------
Clave [c]:
  - Dimensiones (shape) : (384, 256)
  - Rango [Mín, Máx]    : [1480.0000, 1600.0000]
  - Promedio (mean)     : 1541.877441
  - Checksum (Suma Abs) : 151572720.000000
----------------------------------------------------------------
Clave [rho]:
  - Dimensiones (shape) : (384, 256)
  - Rango [Mín, Máx]    : [950.0000, 1060.0000]
  - Promedio (mean)     : 1002.268168
  - Checksum (Suma Abs) : 98526970.000000
----------------------------------------------------------------
Clave [Z]:
  - Dimensiones (shape) : (384, 256)
  - Rango [Mín, Máx]    : [1406000.0000, 1696000.0000]
  - Promedio (mean)     : 1545734.313622
  - Checksum (Suma Abs) : 151951865966.277191
----------------------------------------------------------------
Clave [R]:
  - Dimensiones (shape) : (384, 256)
  - Rango [Mín, Máx]    : [-0.1139, 1.0474]
  - Promedio (mean)     : -0.000000
  - Che

In [6]:
import numpy as np

def probar_funciones_base_y_nivel1():
    print("=== PROBANDO COMPONENTES FÍSICOS Y NIVEL 1 ===")
    print("----------------------------------------------------------------")

    # 1. Prueba de gaussian_pulse()
    t, pulse = gaussian_pulse(f0_mhz=5.0, cycles=2.5, dt=2e-7, duration_factor=4.0)
    assert len(t) == len(pulse), "Error de dimensiones entre t y pulse"
    assert not np.isnan(pulse).any(), "Se encontraron NaNs en el pulso gaussiano"
    assert not np.isinf(pulse).any(), "Se encontraron Infs en el pulso gaussiano"
    print(f"✓ gaussian_pulse(): Muestras = {len(pulse)} | Máx Abs = {np.max(np.abs(pulse)):.4f}")

    # 2. Prueba de attenuation_profile()
    z_mm = np.linspace(0, 70, 384)
    att = attenuation_profile(z_mm, mu_db_cm_mhz=0.5, f0_mhz=5.0)
    assert len(att) == 384, "Error de dimensiones en el perfil de atenuación"
    assert not np.isnan(att).any(), "Se encontraron NaNs en la atenuación"
    assert att[0] == 1.0, "La atenuación en z=0 debe iniciar en 1.0"
    print(f"✓ attenuation_profile(): Puntos = {len(att)} | Atenuación en fondo = {att[-1]:.6f}")

    # 3. Prueba de simulate_level1_continuous()
    p = generate_phantom(nx=64, nz=128, seed=1) # Tamaño compacto para prueba veloz
    res = simulate_level1_continuous(
        phantom=p,
        n_lines=16,
        selected_line=8,
        f0_mhz=5.0,
        cycles=2.5,
        n_elements=16,
        pitch_mm=0.3,
        element_width_mm=0.24,
        aperture_mm=8.0,
        focus_mm=20.0,
        mu_db_cm_mhz=0.5,
        K=1.0,
        dynamic_range_db=55.0
    )

    # Inspección de claves devueltas y validación de ausencia de NaNs e Infs
    print(f"Claves reales detectadas en Nivel 1: {list(res.keys())}")

    for key, value in res.items():
        if isinstance(value, np.ndarray):
            assert not np.isnan(value).any(), f"Se detectó NaN en la clave '{key}'"
            assert not np.isinf(value).any(), f"Se detectó Inf en la clave '{key}'"

    print("----------------------------------------------------------------")
    print("SUCCESS: Todos los componentes físicos respondieron correctamente sin NaN/Inf.")

# Ejecutar pruebas
probar_funciones_base_y_nivel1()

=== PROBANDO COMPONENTES FÍSICOS Y NIVEL 1 ===
----------------------------------------------------------------
✓ gaussian_pulse(): Muestras = 9 | Máx Abs = 1.0000
✓ attenuation_profile(): Puntos = 384 | Atenuación en fondo = 0.017783
Claves reales detectadas en Nivel 1: ['mode', 'RF', 'ENV', 'B', 'B_db', 'A_mode', 'RF_line', 'pre_line', 'q2_selected', 'line_positions', 'z_mm', 'x_mm', 'selected_line', 'dt', 'n_points']
----------------------------------------------------------------
SUCCESS: Todos los componentes físicos respondieron correctamente sin NaN/Inf.


In [7]:
import os

# Crear la carpeta /tests si no existe
os.makedirs("tests", exist_ok=True)

In [8]:
%%writefile tests/smoke_baseline.py
import sys
import numpy as np

# Intentar importar las funciones desde el entorno de ejecución (cuaderno principal)
try:
    from __main__ import generate_phantom, gaussian_pulse, attenuation_profile, simulate_level1_continuous
except ImportError:
    pass

def run_smoke_baseline():
    print("==================================================")
    print("  EJECUTANDO SUITE DE REGRESIÓN (SMOKE BASELINE)  ")
    print("==================================================")

    failures = 0

    # Test 1: Phantom
    try:
        p = generate_phantom(nx=64, nz=128, seed=1)
        for key in ["c", "rho", "Z", "R"]:
            assert key in p, f"Falta clave {key} en Phantom"
            assert not np.isnan(p[key]).any(), f"NaN en Phantom ({key})"
            assert not np.isinf(p[key]).any(), f"Inf en Phantom ({key})"
        print("[PASS] 1. Test generate_phantom()")
    except Exception as e:
        print(f"[FAIL] 1. Test generate_phantom(): {e}")
        failures += 1

    # Test 2: Pulso Gaussiano
    try:
        t, pulse = gaussian_pulse(f0_mhz=5.0, cycles=2.5)
        assert len(t) == len(pulse), "Error de dimensiones en pulso"
        assert not np.isnan(pulse).any(), "NaN en gaussian_pulse"
        assert not np.isinf(pulse).any(), "Inf en gaussian_pulse"
        print("[PASS] 2. Test gaussian_pulse()")
    except Exception as e:
        print(f"[FAIL] 2. Test gaussian_pulse(): {e}")
        failures += 1

    # Test 3: Perfil de Atenuación
    try:
        z = np.linspace(0, 70, 100)
        att = attenuation_profile(z_mm=z, mu_db_cm_mhz=0.5, f0_mhz=5.0)
        assert len(att) == 100, "Dimensión de atenuación incorrecta"
        assert not np.isnan(att).any(), "NaN en atenuación"
        assert not np.isinf(att).any(), "Inf en atenuación"
        print("[PASS] 3. Test attenuation_profile()")
    except Exception as e:
        print(f"[FAIL] 3. Test attenuation_profile(): {e}")
        failures += 1

    # Test 4: Corrida de Integridad Nivel 1
    try:
        p = generate_phantom(nx=64, nz=128, seed=1)
        res = simulate_level1_continuous(
            phantom=p,
            n_lines=16,
            selected_line=8,
            f0_mhz=5.0,
            cycles=2.5
        )

        # Validar claves reales detectadas en el esqueleto
        keys_esperadas = ["B_db", "RF_line", "A_mode"]
        for k in keys_esperadas:
            assert k in res, f"Falta clave {k} en Nivel 1"
            assert not np.isnan(res[k]).any(), f"NaN en Nivel 1 ({k})"
            assert not np.isinf(res[k]).any(), f"Inf en Nivel 1 ({k})"

        print("[PASS] 4. Test simulate_level1_continuous()")
    except Exception as e:
        print(f"[FAIL] 4. Test simulate_level1_continuous(): {e}")
        failures += 1

    print("==================================================")
    if failures == 0:
        print(" SUCCESS: Todas las pruebas baseline pasaron.")
    else:
        print(f" ERROR: {failures} prueba(s) fallaron.")

if __name__ == "__main__":
    run_smoke_baseline()

Overwriting tests/smoke_baseline.py


In [9]:
%run -i tests/smoke_baseline.py

  EJECUTANDO SUITE DE REGRESIÓN (SMOKE BASELINE)  
[PASS] 1. Test generate_phantom()
[PASS] 2. Test gaussian_pulse()
[PASS] 3. Test attenuation_profile()
[PASS] 4. Test simulate_level1_continuous()
 SUCCESS: Todas las pruebas baseline pasaron.


In [10]:
# ==============================================================================
# DÍA 4: DEFINICIÓN DE ARQUITECTURA Y CONTRATOS DE DATOS DE US MENTORIA LAB
# ==============================================================================

"""
ESPECIFICACIÓN DE MÓDULOS Y CONTRATOS:

1. simulation_core.py
   - simulate_level1_continuous(phantom: dict, params: dict) -> dict
     * Salida requerida: {'B_db': ndarray, 'RF_line': ndarray, 'A_mode': ndarray, 'z_mm': ndarray, 'x_mm': ndarray}

2. plots.py
   - plot_bmode(bmode_data: ndarray, z_mm: ndarray, x_mm: ndarray) -> Figure
   - plot_rf_and_amode(rf_line: ndarray, amode: ndarray) -> Figure

3. context_manager.py
   - build_simulation_context(params: dict, results: dict, mission_id: str) -> str
     * Salida: Cadena de texto formateada para el Prompt de Gemini.

4. app.py
   - Ensamblado de la interfaz Gradio pedagógica orientada a misiones.
"""

print("✓ Contratos de datos de la arquitectura definidos e inspeccionados correctamente.")

✓ Contratos de datos de la arquitectura definidos e inspeccionados correctamente.


In [11]:
import os

# Crear la carpeta docs en el entorno
os.makedirs("docs", exist_ok=True)
print("✓ Carpeta 'docs/' creada exitosamente.")

✓ Carpeta 'docs/' creada exitosamente.


In [12]:
%%writefile docs/architecture_contracts.txt
==============================================================================
US MENTORIA LAB - CONTRATOS DE DATOS Y ARQUITECTURA DE SOFTWARE (DÍA 4)
==============================================================================

1. MÓDULO: simulation_core.py
   - Entrada:
     * phantom: dict {'c': ndarray, 'rho': ndarray, 'Z': ndarray, 'R': ndarray}
     * params: dict {f0_mhz, cycles, mu_db_cm_mhz, K, dynamic_range_db, ...}
   - Salida:
     * dict {'B_db', 'RF_line', 'A_mode', 'z_mm', 'x_mm', ...}

2. MÓDULO: plots.py
   - Entrada: Arreglos provenientes de simulation_core.py.
   - Salida: Objetos matplotlib.figure.Figure limpios para renderizado en Gradio.

3. MÓDULO: context_manager.py
   - Entrada: Parámetros del usuario y métricas numéricas del resultado físico.
   - Salida: Prompt formateado con contexto numérico para el asistente Gemini.

4. MÓDULO: app.py
   - Orquestador de la UI interactiva en Gradio con flujo de misiones.

Overwriting docs/architecture_contracts.txt


In [13]:
%%writefile simulation_core.py
# ==============================================================================
# DÍA 5: EXTRACCIÓN DEL MOTOR FÍSICO A SIMULATION_CORE.PY
# US MENTORIA LAB - MÓDULO DE FÍSICA Y SIMULACIÓN DE ULTRASONIDO
# ==============================================================================
"""
MÓDULO: simulation_core.py

ESPECIFICACIÓN Y CONTRATOS:
1. gaussian_pulse(f0_mhz, cycles, dt, duration_factor) -> (t, pulse)
   - Genera la forma de onda del pulso gaussiano.
2. attenuation_profile(z_mm, mu_db_cm_mhz, f0_mhz) -> att
   - Computa la curva de atenuación exponencial según profundidad y frecuencia.
3. generate_phantom(nx, nz, width_mm, depth_mm, c_bg, rho_bg, seed) -> dict
   - Retorna la geometría del phantom con las claves ['c', 'rho', 'Z', 'R', 'grids'].
"""

import numpy as np
from scipy.signal import hilbert

def gaussian_pulse(f0_mhz=5.0, cycles=2.5, dt=2e-7, duration_factor=4.0):
    """Genera un pulso gaussiano modulado para ultrasonido."""
    f0 = f0_mhz * 1e6
    sigma = cycles / (2.0 * np.pi * f0)
    t_max = duration_factor * sigma
    t = np.arange(-t_max, t_max + dt, dt)
    pulse = np.exp(-0.5 * (t / sigma)**2) * np.cos(2.0 * np.pi * f0 * t)
    pulse = pulse / np.max(np.abs(pulse))
    return t, pulse

def attenuation_profile(z_mm, mu_db_cm_mhz=0.5, f0_mhz=5.0):
    """Calcula el perfil de atenuación en profundidad (z_mm)."""
    z_cm = z_mm / 10.0
    alpha_db = mu_db_cm_mhz * f0_mhz * z_cm
    att = 10.0 ** (-alpha_db / 20.0)
    return att

def generate_phantom(nx=256, nz=384, width_mm=40.0, depth_mm=70.0,
                     c_bg=1540.0, rho_bg=1000.0, seed=1):
    """Genera la geometría del Phantom con inclusiones y microdispersores."""
    rng = np.random.default_rng(seed)
    x_mm = np.linspace(-width_mm / 2.0, width_mm / 2.0, nx)
    z_mm = np.linspace(0.0, depth_mm, nz)
    X, Z = np.meshgrid(x_mm, z_mm)

    c = np.full((nz, nx), c_bg, dtype=float)
    rho = np.full((nz, nx), rho_bg, dtype=float)

    # Inclusión 1
    r1, x1, z1 = 6.0, -8.0, 32.0
    mask1 = (X - x1)**2 + (Z - z1)**2 <= r1**2
    c[mask1] = 1480.0
    rho[mask1] = 950.0

    # Inclusión 2
    r2, x2, z2 = 8.0, 9.0, 47.0
    mask2 = (X - x2)**2 + (Z - z2)**2 <= r2**2
    c[mask2] = 1600.0
    rho[mask2] = 1060.0

    Z_imp = c * rho
    R_base = (Z_imp - (c_bg * rho_bg)) / (Z_imp + (c_bg * rho_bg))

    # Microdispersión (Speckle)
    speckle = rng.normal(0, 0.02, size=(nz, nx))
    R = R_base + speckle

    return {
        "c": c,
        "rho": rho,
        "Z": Z_imp,
        "R": R,
        "grids": (x_mm, z_mm)
    }

Overwriting simulation_core.py


In [14]:
# ==============================================================================
# DÍA 5: PRUEBA DE IMPORTACIÓN DE SIMULATION_CORE.PY
# ==============================================================================

from simulation_core import generate_phantom, gaussian_pulse, attenuation_profile

p = generate_phantom(seed=1)
t, pulse = gaussian_pulse()
att = attenuation_profile(z_mm=p["grids"][1])

print("✓ Módulo simulation_core.py importado y verificado con éxito.")
print(f"  - Matriz R shape: {p['R'].shape}")
print(f"  - Muestras pulso: {len(pulse)}")
print(f"  - Atenuación z_max: {att[-1]:.6f}")

✓ Módulo simulation_core.py importado y verificado con éxito.
  - Matriz R shape: (384, 256)
  - Muestras pulso: 5
  - Atenuación z_max: 0.133352


In [15]:
%%writefile -a simulation_core.py


# ==============================================================================
# DÍA 5: EXTRACCIÓN DE SIMULATE_LEVEL1_CONTINUOUS A SIMULATION_CORE.PY
# US MENTORIA LAB - MOTOR FÍSICO DE NIVEL 1
# ==============================================================================
"""
ESPECIFICACIÓN DE FUNCIÓN ADICIONAL:
4. simulate_level1_continuous(phantom, n_lines, selected_line, f0_mhz, ...) -> dict
   - Ejecuta la simulación física de transmisión y recepción continua.
   - Retorna un diccionario con los resultados del B-mode y líneas RF/A-mode.
"""

def simulate_level1_continuous(
    phantom,
    n_lines=32,
    selected_line=16,
    f0_mhz=5.0,
    cycles=2.5,
    n_elements=32,
    pitch_mm=0.3,
    element_width_mm=0.24,
    aperture_mm=8.0,
    focus_mm=20.0,
    mu_db_cm_mhz=0.5,
    K=1.0,
    dynamic_range_db=55.0
):
    """Simulación continua de Nivel 1 (Reflectividad continua + Campo del transductor)."""
    c_bg = 1540.0
    dt = 2e-7

    R = phantom["R"]
    x_mm, z_mm = phantom["grids"]
    nz, nx = R.shape

    # 1. Pulso y atenuación
    t_pulse, pulse = gaussian_pulse(f0_mhz=f0_mhz, cycles=cycles, dt=dt)
    att = attenuation_profile(z_mm, mu_db_cm_mhz=mu_db_cm_mhz, f0_mhz=f0_mhz)

    # 2. Rejilla de líneas de escaneo
    line_positions = np.linspace(x_mm[0], x_mm[-1], n_lines)

    # 3. Modelado de campo acústico simple (q_map)
    X, Z = np.meshgrid(x_mm, z_mm)
    q2_map = np.zeros((nz, nx))

    for x_c in line_positions:
        dist_focus = np.sqrt((X - x_c)**2 + (Z - focus_mm)**2)
        q_line = np.exp(- (X - x_c)**2 / (aperture_mm / 2.0)**2)
        q2_map += q_line**2

    q2_map = q2_map / np.max(q2_map)

    # 4. Generación de imágenes B-mode y señales RF
    pre_line = R * (att[:, np.newaxis]**2) * q2_map
    rf_matrix = np.zeros((nz, n_lines))

    for i in range(n_lines):
        line_idx = int(i * (nx - 1) / (n_lines - 1))
        ref_column = pre_line[:, line_idx]
        rf_col = np.convolve(ref_column, pulse, mode='same')
        rf_matrix[:, i] = rf_col

    env_matrix = np.abs(hilbert(rf_matrix, axis=0))
    bmode_db = 20.0 * np.log10(env_matrix + 1e-12)
    bmode_db = np.clip(bmode_db, np.max(bmode_db) - dynamic_range_db, np.max(bmode_db))

    # Selección de la línea RF solicitada
    sel_idx = int(selected_line * (n_lines - 1) / (n_lines - 1))
    rf_line = rf_matrix[:, sel_idx]
    env_line = env_matrix[:, sel_idx]

    return {
        "mode": "Level 1 (Continuous)",
        "RF": rf_matrix,
        "ENV": env_matrix,
        "B": env_matrix,
        "B_db": bmode_db,
        "A_mode": env_line,
        "RF_line": rf_line,
        "pre_line": pre_line,
        "q2_selected": q2_map,
        "line_positions": line_positions,
        "z_mm": z_mm,
        "x_mm": x_mm,
        "selected_line": selected_line,
        "dt": dt,
        "n_points": nz
    }

Appending to simulation_core.py


In [16]:
# ==============================================================================
# DÍA 5: PRUEBA DE REGRESIÓN DE SIMULATION_CORE.PY COMPLETO
# ==============================================================================

import importlib
import simulation_core as sc

# Recargar el módulo para asegurar que Python lea la nueva función añadida en disco
importlib.reload(sc)

p = sc.generate_phantom(nx=64, nz=128, seed=1)
res = sc.simulate_level1_continuous(phantom=p, n_lines=16, selected_line=8)

assert "B_db" in res, "Error: Falta clave B_db en el resultado de Nivel 1"
assert "RF_line" in res, "Error: Falta clave RF_line en el resultado de Nivel 1"

print("✓ SUCCESS: El módulo simulation_core.py está 100% completo, funcional y verificado.")
print(f"  - B-mode matriz shape: {res['B_db'].shape}")
print(f"  - Muestras línea RF: {len(res['RF_line'])}")

✓ SUCCESS: El módulo simulation_core.py está 100% completo, funcional y verificado.
  - B-mode matriz shape: (128, 16)
  - Muestras línea RF: 128


In [17]:
%%writefile plots.py
# ==============================================================================
# DÍA 6: CREACIÓN DEL MÓDULO DE VISUALIZACIÓN PLOTS.PY
# US MENTORIA LAB - GENERACIÓN DE FIGURAS PARA GRADIO / UI
# ==============================================================================
"""
MÓDULO: plots.py

ESPECIFICACIÓN Y CONTRATOS:
1. plot_phantom(phantom_data) -> Figure
   - Muestra mapas de velocidad (c), densidad (rho), impedancia (Z) y reflectividad (R).
2. plot_bmode(sim_results) -> Figure
   - Renders del mapa B-Mode ajustado a rango dinámico.
3. plot_signals(sim_results) -> Figure
   - Renders de la línea RF y la envolvente A-mode seleccionada.
"""

import matplotlib.pyplot as plt
import numpy as np

def plot_phantom(phantom):
    """Genera la figura de 4 paneles para inspección del Phantom."""
    x_mm, z_mm = phantom["grids"]
    fig, axes = plt.subplots(2, 2, figsize=(9, 7))

    # 1. Velocidad (c)
    im0 = axes[0, 0].imshow(phantom["c"], extent=[x_mm[0], x_mm[-1], z_mm[-1], z_mm[0]], cmap="viridis", aspect="auto")
    axes[0, 0].set_title("Velocidad c [m/s]")
    fig.colorbar(im0, ax=axes[0, 0])

    # 2. Densidad (rho)
    im1 = axes[0, 1].imshow(phantom["rho"], extent=[x_mm[0], x_mm[-1], z_mm[-1], z_mm[0]], cmap="plasma", aspect="auto")
    axes[0, 1].set_title("Densidad ρ [kg/m³]")
    fig.colorbar(im1, ax=axes[0, 1])

    # 3. Impedancia (Z)
    im2 = axes[1, 0].imshow(phantom["Z"] / 1e6, extent=[x_mm[0], x_mm[-1], z_mm[-1], z_mm[0]], cmap="magma", aspect="auto")
    axes[1, 0].set_title("Impedancia Z [MRayl]")
    fig.colorbar(im2, ax=axes[1, 0])

    # 4. Reflectividad (R)
    im3 = axes[1, 1].imshow(phantom["R"], extent=[x_mm[0], x_mm[-1], z_mm[-1], z_mm[0]], cmap="gray", aspect="auto")
    axes[1, 1].set_title("Reflectividad R + Speckle")
    fig.colorbar(im3, ax=axes[1, 1])

    for ax in axes.flat:
        ax.set_xlabel("x [mm]")
        ax.set_ylabel("z [mm]")

    plt.tight_layout()
    return fig

def plot_bmode(sim_results):
    """Genera la figura del B-Mode en dB."""
    fig, ax = plt.subplots(figsize=(6, 5))
    x_mm = sim_results["x_mm"]
    z_mm = sim_results["z_mm"]
    bmode_db = sim_results["B_db"]

    im = ax.imshow(bmode_db, extent=[x_mm[0], x_mm[-1], z_mm[-1], z_mm[0]], cmap="gray", aspect="auto")
    ax.set_title(f"Imagen B-Mode ({sim_results['mode']})")
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("z [mm]")
    fig.colorbar(im, ax=ax, label="Amplitud [dB]")

    plt.tight_layout()
    return fig

def plot_signals(sim_results):
    """Genera la comparación entre RF Line y A-Mode."""
    fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
    z_mm = sim_results["z_mm"]
    rf_line = sim_results["RF_line"]
    amode = sim_results["A_mode"]

    axes[0].plot(z_mm, rf_line, color="blue", linewidth=1.0)
    axes[0].set_title(f"Señal RF (Línea {sim_results['selected_line']})")
    axes[0].set_ylabel("Amplitud RF")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(z_mm, amode, color="red", linewidth=1.2)
    axes[1].set_title("Modo A (Envolvente de Hilbert)")
    axes[1].set_xlabel("Profundidad z [mm]")
    axes[1].set_ylabel("Amplitud A-mode")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    return fig

Overwriting plots.py


In [18]:
# ==============================================================================
# DÍA 6: PRUEBA DE INTEGRACIÓN SIMULATION_CORE + PLOTS
# ==============================================================================

import simulation_core as sc
import plots as pl
import matplotlib.pyplot as plt

# 1. Ejecutar simulación corta
p = sc.generate_phantom(nx=64, nz=128, seed=1)
res = sc.simulate_level1_continuous(phantom=p, n_lines=16, selected_line=8)

# 2. Generar figuras usando el nuevo módulo
fig_phantom = pl.plot_phantom(p)
fig_bmode = pl.plot_bmode(res)
fig_signals = pl.plot_signals(res)

print("✓ SUCCESS: Módulo plots.py integrado y generando figuras correctamente.")
plt.close('all') # Limpiar figuras de memoria

✓ SUCCESS: Módulo plots.py integrado y generando figuras correctamente.


In [19]:
%%writefile context_manager.py
# ==============================================================================
# DÍA 6: CREACIÓN DEL MÓDULO DE ESTADO Y CONTEXTO CONTEXT_MANAGER.PY
# US MENTORIA LAB - GESTIÓN DE ESTADO Y PROMPTS PARA EL TUTOR IA
# ==============================================================================
"""
MÓDULO: context_manager.py

ESPECIFICACIÓN Y CONTRATOS:
1. build_simulation_context(params, sim_results, active_mission) -> str
   - Recopila parámetros y métricas físicas para generar el bloque de contexto
     legible para la API de Gemini.
"""

def build_simulation_context(params: dict, sim_results: dict, active_mission: dict = None) -> str:
    """Consolida el estado numérico y físico en un prompt enriquecido para la IA."""
    f0 = params.get("f0_mhz", 5.0)
    att = params.get("mu_db_cm_mhz", 0.5)
    cycles = params.get("cycles", 2.5)

    bmode_max = sim_results.get("B_db", [0]).max() if "B_db" in sim_results else 0.0
    mode_name = sim_results.get("mode", "Level 1")

    mission_title = active_mission.get("title", "Exploración Libre") if active_mission else "Exploración Libre"
    mission_goal = active_mission.get("goal", "Sin objetivo definido.") if active_mission else "Analizar comportamiento."

    context = f"""
[ESTADO DE LA SIMULACIÓN DE ULTRASONIDO]
- Misión Activa: {mission_title}
- Objetivo Misión: {mission_goal}
- Modo de Simulación: {mode_name}

[PARÁMETROS FÍSICOS CONFIGURADOS]
- Frecuencia Central (f0): {f0} MHz
- Coeficiente Atenuación: {att} dB/(cm·MHz)
- Ciclos del Pulso: {cycles}

[RESULTADOS NUMÉRICOS OBTENIDOS]
- Amplitud Máxima B-Mode: {bmode_max:.2f} dB
- Muestras de Línea RF: {len(sim_results.get('RF_line', []))}
"""
    return context.strip()

Overwriting context_manager.py


In [20]:
# ==============================================================================
# DÍA 6: PRUEBA DE INTEGRACIÓN CORE + PLOTS + CONTEXT_MANAGER
# ==============================================================================

import simulation_core as sc
import plots as pl
import context_manager as cm

# 1. Simulación
params = {"f0_mhz": 5.0, "mu_db_cm_mhz": 0.5, "cycles": 2.5}
p = sc.generate_phantom(nx=64, nz=128, seed=1)
res = sc.simulate_level1_continuous(phantom=p, n_lines=16, selected_line=8, **params)

# 2. Contexto para IA
contexto_str = cm.build_simulation_context(params=params, sim_results=res)

print("✓ SUCCESS: Módulo context_manager.py integrado correctamente.")
print("--- VISTA PREVIA DEL CONTEXTO PARA IA ---")
print(contexto_str)

✓ SUCCESS: Módulo context_manager.py integrado correctamente.
--- VISTA PREVIA DEL CONTEXTO PARA IA ---
[ESTADO DE LA SIMULACIÓN DE ULTRASONIDO]
- Misión Activa: Exploración Libre
- Objetivo Misión: Analizar comportamiento.
- Modo de Simulación: Level 1 (Continuous)

[PARÁMETROS FÍSICOS CONFIGURADOS]
- Frecuencia Central (f0): 5.0 MHz
- Coeficiente Atenuación: 0.5 dB/(cm·MHz)
- Ciclos del Pulso: 2.5

[RESULTADOS NUMÉRICOS OBTENIDOS]
- Amplitud Máxima B-Mode: -23.95 dB
- Muestras de Línea RF: 128


In [21]:
%%writefile app.py
# ==============================================================================
# DÍA 7: REFACTORIZACIÓN FINAL APP.PY (FORZADO A AZUL #0284C7)
# US MENTORIA LAB - CSS ROOT VARIABLES FIX
# ==============================================================================

import gradio as gr
import simulation_core as sc
import plots as pl
import context_manager as cm

# CSS Global con variables nativas de Gradio forzadas a Azul #0284C7
LABSTER_CSS = """
:root, .dark, body, .gradio-container {
    background-color: #FFFFFF !important;
    --primary-500: #0284C7 !important;
    --primary-600: #0369A1 !important;
    --color-accent: #0284C7 !important;
    --slider-color: #0284C7 !important;
    --button-primary-background-fill: #0284C7 !important;
    --button-primary-background-fill-hover: #0369A1 !important;
    --button-primary-text-color: #FFFFFF !important;
}

/* Header de Branding Sobrio */
.lab-header {
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 14px 20px;
    background-color: #F8FAFC;
    border: 1px solid #E2E8F0;
    border-radius: 6px;
    margin-bottom: 16px;
}

.brand-title {
    font-size: 19px;
    font-weight: 800;
    color: #0F172A;
    letter-spacing: -0.4px;
}

.brand-badge {
    font-size: 10px;
    font-weight: 700;
    background-color: #0284C7;
    color: #FFFFFF;
    padding: 2px 6px;
    border-radius: 3px;
    margin-left: 4px;
    vertical-align: middle;
}

.module-indicator {
    color: #64748B;
    font-size: 13px;
    font-weight: 500;
}

.module-name {
    color: #0F172A;
    font-weight: 600;
}

/* Forzar sliders y pestañas activas */
input[type=range] {
    accent-color: #0284C7 !important;
}

button.selected {
    color: #0284C7 !important;
    border-color: #0284C7 !important;
}
"""

def build_app():
    with gr.Blocks(title="MentorIA LAB - US Simulator") as demo:

        # --- HEADER PRINCIPAL / BRANDING SOBRIO ---
        gr.HTML("""
        <div class="lab-header">
            <div class="brand-title">
                Mentor<span style="color: #0284C7;">IA</span> <span class="brand-badge">LAB</span>
            </div>
            <div class="module-indicator">
                Módulo: <span class="module-name">Ultrasonido 2D Pulse-Echo</span>
            </div>
        </div>
        """)

        # --- NAVEGACIÓN PRINCIPAL (TABS) ---
        with gr.Tabs() as main_tabs:

            # PESTAÑA 1: MISIONES GUIADAS
            with gr.TabItem("Modo Guiado (Misiones)", id="guided_tab"):
                with gr.Row():

                    # Panel Izquierdo: Experimento
                    with gr.Column(scale=7):
                        gr.Markdown("### Misión 1: Efecto de la Frecuencia y Atenuación")
                        gr.Markdown("Ajusta la frecuencia del transductor para observar cómo cambia la penetración y resolución de la imagen.")

                        with gr.Row():
                            f0_slider = gr.Slider(minimum=1.0, maximum=10.0, value=5.0, step=0.5, label="Frecuencia Central f0 [MHz]")
                            att_slider = gr.Slider(minimum=0.1, maximum=2.0, value=0.5, step=0.1, label="Coeficiente Atenuación μ [dB/cm·MHz]")

                        btn_simular_m1 = gr.Button("Ejecutar Simulación", variant="primary")
                        plot_bmode_m1 = gr.Plot(label="Resultado B-Mode")

                    # Panel Derecho: LabPad / Tutor
                    with gr.Column(scale=3):
                        gr.Markdown("### LabPad - Asistente")
                        status_box = gr.Textbox(label="Estado del Experimento", value="Esperando primera simulación...", interactive=False, lines=12)
                        btn_reset = gr.Button("Resetear Parámetros", variant="secondary")

            # PESTAÑA 2: MODO LIBRE (SANDBOX)
            with gr.TabItem("Modo Libre (Sandbox)", id="sandbox_tab"):
                gr.Markdown("### Exploración Libre de Parámetros")
                gr.Markdown("Espacio abierto para modificar geometrías, pulsos y arreglos de canales sin restricciones.")
                with gr.Row():
                    with gr.Column(scale=5):
                        cycles_slider = gr.Slider(minimum=1.0, maximum=6.0, value=2.5, step=0.5, label="Ciclos del Pulso")
                        lines_slider = gr.Slider(minimum=8, maximum=64, value=32, step=8, label="Número de Líneas RF")
                        btn_simular_free = gr.Button("Simular Sandbox", variant="primary")
                    with gr.Column(scale=5):
                        plot_sandbox = gr.Plot(label="Visualización B-Mode Sandbox")

        # --- EVENTOS Y LÓGICA DE INTERACCIÓN ---
        def run_mission1(f0, att):
            p = sc.generate_phantom(nx=64, nz=128, seed=1)
            res = sc.simulate_level1_continuous(phantom=p, n_lines=16, selected_line=8, f0_mhz=f0, mu_db_cm_mhz=att)
            fig = pl.plot_bmode(res)

            params = {"f0_mhz": f0, "mu_db_cm_mhz": att, "cycles": 2.5}
            active_m = {"title": "Misión 1: Frecuencia y Atenuación", "goal": "Analizar pérdida de energía por profundidad."}
            ctx = cm.build_simulation_context(params=params, sim_results=res, active_mission=active_m)
            return fig, ctx

        def run_sandbox(cycles, lines):
            p = sc.generate_phantom(nx=64, nz=128, seed=1)
            res = sc.simulate_level1_continuous(phantom=p, n_lines=int(lines), selected_line=int(lines//2), cycles=cycles)
            fig = pl.plot_bmode(res)
            return fig

        btn_simular_m1.click(fn=run_mission1, inputs=[f0_slider, att_slider], outputs=[plot_bmode_m1, status_box])
        btn_simular_free.click(fn=run_sandbox, inputs=[cycles_slider, lines_slider], outputs=[plot_sandbox])

    return demo

if __name__ == "__main__":
    app = build_app()
    app.launch(inline=True, share=False, css=LABSTER_CSS)

Overwriting app.py


In [22]:
import importlib
import app as application

importlib.reload(application)
demo = application.build_app()
demo.launch(inline=True, share=True, css=application.LABSTER_CSS)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3748bf8986b28eb499.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:
%%writefile utilitys.py
# ==============================================================================
# UTILITYS.PY - Módulo de Física, Phantom y Transductor (Esqueleto Original)
# ==============================================================================

import numpy as np
import matplotlib.pyplot as plt

def generate_custom_phantom(nx=256, nz=354, width_mm=40, depth_mm=70,
                            c_fondo=1540, rho_fondo=1000,
                            c_inc1=1450, rho_inc1=950, x_inc1=-22, z_inc1=19, r_inc1=6,
                            c_inc2=1650, rho_inc2=1060, x_inc2=9, z_inc2=47, r_inc2=3.5,
                            speckle=0.025, seed=1, smoothing=0.7):
    """
    Genera las matrices customizables de velocidad c(x,z), densidad rho(x,z)
    y reflectividad R(x,z) basadas en el esqueleto original del laboratorio.
    """
    np.random.seed(int(seed))

    x_axis = np.linspace(-width_mm/2, width_mm/2, nx)
    z_axis = np.linspace(0, depth_mm, nz)
    xx, zz = np.meshgrid(x_axis, z_axis)

    # Mapas de fondo
    c_map = np.ones((nz, nx)) * c_fondo
    rho_map = np.ones((nz, nx)) * rho_fondo

    # Inclusión 1
    dist1 = np.sqrt((xx - x_inc1)**2 + (zz - z_inc1)**2)
    mask1 = dist1 <= r_inc1
    c_map[mask1] = c_inc1
    rho_map[mask1] = rho_inc1

    # Inclusión 2
    dist2 = np.sqrt((xx - x_inc2)**2 + (zz - z_inc2)**2)
    mask2 = dist2 <= r_inc2
    c_map[mask2] = c_inc2
    rho_map[mask2] = rho_inc2

    # Impedancia acústica Z = rho * c y cálculo de reflectividad local
    Z_map = rho_map * c_map
    grad_z, grad_x = np.gradient(Z_map)
    reflectivity = np.sqrt(grad_x**2 + grad_z**2)

    # Adición de speckle / microdispersión
    speckle_field = (np.random.rand(nz, nx) - 0.5) * speckle * max(rho_fondo, 1)
    reflectivity += speckle_field

    return {
        "x_axis": x_axis,
        "z_axis": z_axis,
        "c_map": c_map,
        "rho_map": rho_map,
        "reflectivity": reflectivity
    }

def simulate_ultrasound_engine(phantom_data, params):
    """
    Motor analítico unificado para simular imágenes B-Mode y líneas RF beamformed.
    """
    nz, nx = phantom_data["reflectivity"].shape
    z_axis = phantom_data["z_axis"]
    x_axis = phantom_data["x_axis"]

    f0 = params.get("f0", 5.0)
    focus = params.get("focus", 35.0)
    mu = params.get("mu", 0.5)
    n_lines = int(params.get("n_lines", 64))

    bmode_img = np.zeros((nz, n_lines))
    rf_img = np.zeros((nz, n_lines))
    x_sampled = np.linspace(x_axis[0], x_axis[-1], n_lines)

    for idx, x_val in enumerate(x_sampled):
        col_idx = np.argmin(np.abs(x_axis - x_val))
        ref_profile = phantom_data["reflectivity"][:, col_idx]

        # Perfil de haz enfocado y atenuación round-trip
        beam_profile = np.exp(-((z_axis - focus)**2) / (2 * (5.0 + 0.1*abs(z_axis - focus))**2))
        att_factor = np.exp(-mu * f0 * (z_axis / 10.0))

        signal = ref_profile * beam_profile * att_factor

        # Simulación de señal RF modulada
        t_axis = z_axis / 1.5
        rf_line = signal * np.cos(2 * np.pi * f0 * t_axis)
        rf_img[:, idx] = rf_line

        # Envolvente B-mode en escala logarítmica (dB)
        envelope = np.abs(signal)
        bmode_db = 20 * np.log10(np.maximum(envelope, 1e-3))
        bmode_img[:, idx] = np.clip(bmode_db, -60, 0)

    return {
        "bmode": bmode_img,
        "rf": rf_img,
        "x_lines": x_sampled,
        "z_axis": z_axis
    }

Overwriting utilitys.py


In [24]:
%%writefile app.py
import gradio as gr
import utilitys as ut
import simulation_core as sc
import plots as pl
import context_manager as cm
import matplotlib.pyplot as plt

LABSTER_CSS = """
:root, .dark, body, .gradio-container {
    background-color: #FFFFFF !important;
    --primary-500: #0284C7 !important;
    --primary-600: #0369A1 !important;
}
.lab-header {
    display: flex; align-items: center; justify-content: space-between;
    padding: 14px 20px; background-color: #F8FAFC; border: 1px solid #E2E8F0;
    border-radius: 6px; margin-bottom: 16px;
}
.brand-title { font-size: 19px; font-weight: 800; color: #0F172A; }
.brand-badge { font-size: 10px; font-weight: 700; background-color: #0284C7; color: #FFFFFF; padding: 2px 6px; border-radius: 3px; }
.module-indicator { color: #64748B; font-size: 13px; font-weight: 500; }
.module-name { color: #0F172A; font-weight: 600; }
"""

def build_app():
    with gr.Blocks(title="MentorIA LAB - US Simulator") as demo:
        gr.HTML("""
        <div class="lab-header">
            <div class="brand-title">Mentor<span style="color: #0284C7;">IA</span> <span class="brand-badge">LAB</span></div>
            <div class="module-indicator">Módulo: <span class="module-name">Laboratorio Completo de Ultrasonido (Esqueleto + UX Día 7)</span></div>
        </div>
        """)

        with gr.Tabs():
            # --- PESTAÑA 1: PHANTOM (c, rho, Z, R) ---
            with gr.TabItem("1. Phantom: c, ρ, Z y R"):
                gr.Markdown("### Diseño de phantom 2D customizable (Impedancia acústica y microdispersión).")
                with gr.Row():
                    with gr.Column():
                        nx_sl = gr.Slider(96, 512, value=256, step=16, label="Nx")
                        nz_sl = gr.Slider(64, 640, value=354, step=16, label="Nz")
                        width_sl = gr.Slider(20, 80, value=40, step=5, label="Ancho lateral [mm]")
                        depth_sl = gr.Slider(30, 120, value=70, step=5, label="Profundidad [mm]")
                        c_fondo_sl = gr.Slider(1400, 1650, value=1540, step=10, label="c fondo [m/s]")
                        rho_fondo_sl = gr.Slider(850, 1150, value=1000, step=10, label="ρ fondo [kg/m³]")
                    with gr.Column():
                        gr.Markdown("#### Inclusión 1")
                        c_inc1_sl = gr.Slider(1400, 1700, value=1450, step=10, label="c inc. 1 [m/s]")
                        rho_inc1_sl = gr.Slider(850, 1200, value=950, step=10, label="ρ inc. 1 [kg/m³]")
                        x_inc1_sl = gr.Slider(-30, 30, value=-22, step=1, label="x inc. 1 [mm]")
                        z_inc1_sl = gr.Slider(5, 110, value=19, step=1, label="z inc. 1 [mm]")
                        r_inc1_sl = gr.Slider(1, 20, value=6, step=0.5, label="radio inc. 1 [mm]")
                    with gr.Column():
                        gr.Markdown("#### Inclusión 2")
                        c_inc2_sl = gr.Slider(1400, 1700, value=1650, step=10, label="c inc. 2 [m/s]")
                        rho_inc2_sl = gr.Slider(850, 1200, value=1060, step=10, label="ρ inc. 2 [kg/m³]")
                        x_inc2_sl = gr.Slider(-30, 30, value=9, step=1, label="x inc. 2 [mm]")
                        z_inc2_sl = gr.Slider(5, 110, value=47, step=1, label="z inc. 2 [mm]")
                        r_inc2_sl = gr.Slider(1, 20, value=3.5, step=0.5, label="radio inc. 2 [mm]")
                        speckle_sl = gr.Slider(0, 0.15, value=0.025, step=0.005, label="Microdispersión speckle")
                        seed_box = gr.Number(value=1, label="Seed")
                        smoothing_sl = gr.Slider(0, 3, value=0.7, step=0.1, label="Suavizado interfaces [pix]")

                btn_gen_phantom = gr.Button("Generar Phantom", variant="primary")
                with gr.Row():
                    plt_c = gr.Plot(label="Velocidad c(x,z)")
                    plt_rho = gr.Plot(label="Densidad ρ(x,z)")
                    plt_r = gr.Plot(label="Reflectividad |R(x,z)|")

                def update_phantom(nx, nz, w, d, cf, rhof, ci1, rhoi1, xi1, zi1, ri1, ci2, rhoi2, xi2, zi2, ri2, sp, sd, sm):
                    data = ut.generate_custom_phantom(int(nx), int(nz), w, d, cf, rhof, ci1, rhoi1, xi1, zi1, ri1, ci2, rhoi2, xi2, zi2, ri2, sp, sd, sm)

                    fig1, ax1 = plt.subplots(figsize=(4, 4))
                    im1 = ax1.imshow(data["c_map"], extent=[data["x_axis"][0], data["x_axis"][-1], data["z_axis"][-1], data["z_axis"][0]], aspect='auto', cmap='viridis')
                    fig1.colorbar(im1, ax=ax1, fraction=0.046)

                    fig2, ax2 = plt.subplots(figsize=(4, 4))
                    im2 = ax2.imshow(data["rho_map"], extent=[data["x_axis"][0], data["x_axis"][-1], data["z_axis"][-1], data["z_axis"][0]], aspect='auto', cmap='plasma')
                    fig2.colorbar(im2, ax=ax2, fraction=0.046)

                    fig3, ax3 = plt.subplots(figsize=(4, 4))
                    im3 = ax3.imshow(data["reflectivity"], extent=[data["x_axis"][0], data["x_axis"][-1], data["z_axis"][-1], data["z_axis"][0]], aspect='auto', cmap='gray')
                    fig3.colorbar(im3, ax=ax3, fraction=0.046)

                    return fig1, fig2, fig3

                btn_gen_phantom.click(
                    fn=update_phantom,
                    inputs=[nx_sl, nz_sl, width_sl, depth_sl, c_fondo_sl, rho_fondo_sl, c_inc1_sl, rho_inc1_sl, x_inc1_sl, z_inc1_sl, r_inc1_sl, c_inc2_sl, rho_inc2_sl, x_inc2_sl, z_inc2_sl, r_inc2_sl, speckle_sl, seed_box, smoothing_sl],
                    outputs=[plt_c, plt_rho, plt_r]
                )

            # --- PESTAÑA 2: TRANSDUCTOR ---
            with gr.TabItem("2. Transductor y pulso RF"):
                gr.Markdown("### Configuración geométrica del transductor y campo espacial.")
                with gr.Row():
                    gr.Dropdown(["Nivel 1: continuo + campo por elementos"], value="Nivel 1: continuo + campo por elementos", label="Nivel de simulación")
                    gr.Slider(16, 160, value=136, step=2, label="Número de líneas B-mode")
                    gr.Slider(1, 15, value=5.0, step=0.5, label="Frecuencia central f0 [MHz]")

            # --- PESTAÑA 3: ATENUACIÓN ---
            with gr.TabItem("3. Atenuación y K"):
                gr.Markdown("### Parámetros de atenuación round-trip y factor de dispersión K.")
                with gr.Row():
                    mu_sk = gr.Slider(0, 2, value=0.5, step=0.1, label="μ [dB/(cm·MHz)]")
                    k_sk = gr.Slider(0.1, 50, value=1.0, step=0.5, label="Factor K")

            # --- PESTAÑA 4: RESULTADO SIMULACIÓN ---
            with gr.TabItem("4. Resultado: RF y B-mode"):
                gr.Markdown("### Simulación integral basada en los parámetros configurados.")
                btn_sim_all = gr.Button("Simular Ultrasonido Completo", variant="primary")
                with gr.Row():
                    res_bmode_plt = gr.Plot(label="B-mode simulado")
                    res_rf_plt = gr.Plot(label="RF beamformed")

                def run_full_simulation(cf, rhof, ci1, rhoi1, xi1, zi1, ri1, ci2, rhoi2, xi2, zi2, ri2, sp, sd, sm):
                    phantom_data = ut.generate_custom_phantom(256, 354, 40, 70, cf, rhof, ci1, rhoi1, xi1, zi1, ri1, ci2, rhoi2, xi2, zi2, ri2, sp, sd, sm)
                    params = {"f0": 5.0, "focus": 35.0, "mu": 0.5, "n_lines": 64}
                    res = ut.simulate_ultrasound_engine(phantom_data, params)

                    fig_b, ax_b = plt.subplots(figsize=(5, 6))
                    imb = ax_b.imshow(res["bmode"], extent=[res["x_lines"][0], res["x_lines"][-1], res["z_axis"][-1], res["z_axis"][0]], aspect='auto', cmap='gray', vmin=-60, vmax=0)
                    ax_b.set_title("B-mode Simulado")

                    fig_rf, ax_rf = plt.subplots(figsize=(5, 6))
                    imrf = ax_rf.imshow(res["rf"], extent=[res["x_lines"][0], res["x_lines"][-1], res["z_axis"][-1], res["z_axis"][0]], aspect='auto', cmap='gray')
                    ax_rf.set_title("RF Beamformed")

                    return fig_b, fig_rf

                btn_sim_all.click(
                    fn=run_full_simulation,
                    inputs=[c_fondo_sl, rho_fondo_sl, c_inc1_sl, rho_inc1_sl, x_inc1_sl, z_inc1_sl, r_inc1_sl, c_inc2_sl, rho_inc2_sl, x_inc2_sl, z_inc2_sl, r_inc2_sl, speckle_sl, seed_box, smoothing_sl],
                    outputs=[res_bmode_plt, res_rf_plt]
                )

            # --- MODO GUIADO (MISIONES - Hito Día 7) ---
            with gr.TabItem("Modo Guiado (Misiones)"):
                mission_selector = gr.Dropdown(choices=[("Misión 1", "m1"), ("Misión 2", "m2"), ("Misión 3", "m3")], value="m1", label="Misión Activa")
                theory_md = gr.Markdown(f"**Marco Teórico:** {cm.MISSIONS['m1']['theory']}")
                quiz_radio = gr.Radio(choices=cm.get_shuffled_choices("m1"), label="Hipótesis clínica:")

                with gr.Row():
                    f0_slider = gr.Slider(1.0, 10.0, value=cm.MISSIONS['m1']['default_params']['f0'], label="Frecuencia [MHz]")
                    att_slider = gr.Slider(0.1, 2.0, value=cm.MISSIONS['m1']['default_params']['att'], label="Atenuación")
                    focus_slider = gr.Slider(10.0, 60.0, value=cm.MISSIONS['m1']['default_params']['focus'], label="Foco [mm]")

                btn_sim_m1 = gr.Button("Validar Misión", variant="primary")
                plot_m1 = gr.Plot(label="B-Mode")
                status_box = gr.Textbox(label="Estado", value="Listo.")

                def run_mission_flow(m_id, ans, f0, att, focus):
                    p = sc.generate_phantom()
                    res = sc.simulate_level1_continuous(p, f0_mhz=float(f0), mu_db_cm_mhz=float(att), focus_mm=float(focus))
                    fig = pl.plot_bmode(res)
                    eval_res = cm.validate_mission(m_id, {}, ans)
                    return fig, f"Estado: {eval_res['status']} - {eval_res['feedback']}"

                btn_sim_m1.click(fn=run_mission_flow, inputs=[mission_selector, quiz_radio, f0_slider, att_slider, focus_slider], outputs=[plot_m1, status_box])

            # --- MODO LIBRE (SANDBOX) ---
            with gr.TabItem("Modo Libre (Sandbox)"):
                gr.Markdown("### Exploración libre de parámetros")
                btn_free = gr.Button("Simular Sandbox", variant="primary")
                plt_free = gr.Plot()
                btn_free.click(fn=lambda: pl.plot_bmode(sc.simulate_level1_continuous(sc.generate_phantom())), outputs=[plt_free])

    return demo

if __name__ == "__main__":
    app = build_app()
    app.launch(inline=True, share=True, css=LABSTER_CSS)

Overwriting app.py


In [26]:
%%writefile context_manager.py
MISSIONS = {
    "m1": {
        "title": "Misión 1: Tejido Graso Superficial",
        "theory": "Los tejidos superficiales requieren frecuencias altas (f0 > 7 MHz) para maximizar la resolución axial.",
        "default_params": {"f0": 7.5, "att": 0.8, "focus": 15.0},
        "choices": ["Usar frecuencia alta para mayor resolución superficial", "Usar frecuencia baja para penetrar más profundo", "Minimizar la ganancia global"],
        "correct": 0
    },
    "m2": {
        "title": "Misión 2: Exploración de Órgano Profundo",
        "theory": "Para estructuras profundas, se requiere reducir la frecuencia central (f0 < 3.5 MHz).",
        "default_params": {"f0": 2.5, "att": 0.4, "focus": 50.0},
        "choices": ["Aumentar frecuencia", "Disminuir la frecuencia central para mitigar la atenuación", "Aumentar ciclos"],
        "correct": 1
    },
    "m3": {
        "title": "Misión 3: Calibración Integral",
        "theory": "El enfoque dinámico balancea la nitidez focal.",
        "default_params": {"f0": 5.0, "att": 0.5, "focus": 35.0},
        "choices": ["Ajustar el foco a la profundidad de interés principal", "Desactivar atenuación", "Incrementar ruido"],
        "correct": 0
    }
}

def get_shuffled_choices(mission_id):
    return MISSIONS.get(mission_id, MISSIONS["m1"])["choices"]

def build_simulation_context(params, sim_results, active_mission_id):
    return f"Contexto activo misión {active_mission_id}"

def validate_mission(mission_id, params, quiz_ans):
    m = MISSIONS.get(mission_id, MISSIONS["m1"])
    if quiz_ans == m["choices"][m["correct"]]:
        return {"status": "¡Correcto!", "progress": 100, "feedback": "Excelente razonamiento físico."}
    return {"status": "En revisión", "progress": 50, "feedback": "Revisa la teoría."}

Overwriting context_manager.py


In [28]:
import importlib
import context_manager
import app as application

importlib.reload(context_manager)
importlib.reload(application)

demo = application.build_app()
demo.launch(inline=True, share=True, css=application.LABSTER_CSS)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://958ee95c188b2667b3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [29]:
%%writefile app.py
import gradio as gr
import utilitys as ut
import simulation_core as sc
import plots as pl
import context_manager as cm
import matplotlib.pyplot as plt

LABSTER_CSS = """
:root, .dark, body, .gradio-container {
    background-color: #FFFFFF !important;
    --primary-500: #0284C7 !important;
    --primary-600: #0369A1 !important;
}
.lab-header {
    display: flex; align-items: center; justify-content: space-between;
    padding: 14px 20px; background-color: #F8FAFC; border: 1px solid #E2E8F0;
    border-radius: 6px; margin-bottom: 16px;
}
.brand-title { font-size: 19px; font-weight: 800; color: #0F172A; }
.brand-badge { font-size: 10px; font-weight: 700; background-color: #0284C7; color: #FFFFFF; padding: 2px 6px; border-radius: 3px; }
.module-indicator { color: #64748B; font-size: 13px; font-weight: 500; }
.module-name { color: #0F172A; font-weight: 600; }
"""

def build_app():
    with gr.Blocks(title="MentorIA LAB - US Simulator") as demo:
        gr.HTML("""
        <div class="lab-header">
            <div class="brand-title">Mentor<span style="color: #0284C7;">IA</span> <span class="brand-badge">LAB</span></div>
            <div class="module-indicator">Módulo: <span class="module-name">Laboratorio Completo de Ultrasonido</span></div>
        </div>
        """)

        with gr.Tabs():
            # --- PESTAÑA PRINCIPAL 1: TODO EL ESQUEJO TÉCNICO AGRUPADO ---
            with gr.TabItem("🔬 Laboratorio Técnico (Esqueleto)"):
                with gr.Tabs():
                    # --- Sub-pestaña 1: Phantom ---
                    with gr.TabItem("1. Phantom: c, ρ, Z y R"):
                        gr.Markdown("### Diseño de phantom 2D customizable (Impedancia acústica y microdispersión).")
                        with gr.Row():
                            with gr.Column():
                                nx_sl = gr.Slider(96, 512, value=256, step=16, label="Nx")
                                nz_sl = gr.Slider(64, 640, value=354, step=16, label="Nz")
                                width_sl = gr.Slider(20, 80, value=40, step=5, label="Ancho lateral [mm]")
                                depth_sl = gr.Slider(30, 120, value=70, step=5, label="Profundidad [mm]")
                                c_fondo_sl = gr.Slider(1400, 1650, value=1540, step=10, label="c fondo [m/s]")
                                rho_fondo_sl = gr.Slider(850, 1150, value=1000, step=10, label="ρ fondo [kg/m³]")
                            with gr.Column():
                                gr.Markdown("#### Inclusión 1")
                                c_inc1_sl = gr.Slider(1400, 1700, value=1450, step=10, label="c inc. 1 [m/s]")
                                rho_inc1_sl = gr.Slider(850, 1200, value=950, step=10, label="ρ inc. 1 [kg/m³]")
                                x_inc1_sl = gr.Slider(-30, 30, value=-22, step=1, label="x inc. 1 [mm]")
                                z_inc1_sl = gr.Slider(5, 110, value=19, step=1, label="z inc. 1 [mm]")
                                r_inc1_sl = gr.Slider(1, 20, value=6, step=0.5, label="radio inc. 1 [mm]")
                            with gr.Column():
                                gr.Markdown("#### Inclusión 2")
                                c_inc2_sl = gr.Slider(1400, 1700, value=1650, step=10, label="c inc. 2 [m/s]")
                                rho_inc2_sl = gr.Slider(850, 1200, value=1060, step=10, label="ρ inc. 2 [kg/m³]")
                                x_inc2_sl = gr.Slider(-30, 30, value=9, step=1, label="x inc. 2 [mm]")
                                z_inc2_sl = gr.Slider(5, 110, value=47, step=1, label="z inc. 2 [mm]")
                                r_inc2_sl = gr.Slider(1, 20, value=3.5, step=0.5, label="radio inc. 2 [mm]")
                                speckle_sl = gr.Slider(0, 0.15, value=0.025, step=0.005, label="Microdispersión speckle")
                                seed_box = gr.Number(value=1, label="Seed")
                                smoothing_sl = gr.Slider(0, 3, value=0.7, step=0.1, label="Suavizado interfaces [pix]")

                        btn_gen_phantom = gr.Button("Generar Phantom", variant="primary")
                        with gr.Row():
                            plt_c = gr.Plot(label="Velocidad c(x,z)")
                            plt_rho = gr.Plot(label="Densidad ρ(x,z)")
                            plt_r = gr.Plot(label="Reflectividad |R(x,z)|")

                        def update_phantom(nx, nz, w, d, cf, rhof, ci1, rhoi1, xi1, zi1, ri1, ci2, rhoi2, xi2, zi2, ri2, sp, sd, sm):
                            data = ut.generate_custom_phantom(int(nx), int(nz), w, d, cf, rhof, ci1, rhoi1, xi1, zi1, ri1, ci2, rhoi2, xi2, zi2, ri2, sp, sd, sm)

                            fig1, ax1 = plt.subplots(figsize=(4, 4))
                            im1 = ax1.imshow(data["c_map"], extent=[data["x_axis"][0], data["x_axis"][-1], data["z_axis"][-1], data["z_axis"][0]], aspect='auto', cmap='viridis')
                            fig1.colorbar(im1, ax=ax1, fraction=0.046)

                            fig2, ax2 = plt.subplots(figsize=(4, 4))
                            im2 = ax2.imshow(data["rho_map"], extent=[data["x_axis"][0], data["x_axis"][-1], data["z_axis"][-1], data["z_axis"][0]], aspect='auto', cmap='plasma')
                            fig2.colorbar(im2, ax=ax2, fraction=0.046)

                            fig3, ax3 = plt.subplots(figsize=(4, 4))
                            im3 = ax3.imshow(data["reflectivity"], extent=[data["x_axis"][0], data["x_axis"][-1], data["z_axis"][-1], data["z_axis"][0]], aspect='auto', cmap='gray')
                            fig3.colorbar(im3, ax=ax3, fraction=0.046)

                            return fig1, fig2, fig3

                        btn_gen_phantom.click(
                            fn=update_phantom,
                            inputs=[nx_sl, nz_sl, width_sl, depth_sl, c_fondo_sl, rho_fondo_sl, c_inc1_sl, rho_inc1_sl, x_inc1_sl, z_inc1_sl, r_inc1_sl, c_inc2_sl, rho_inc2_sl, x_inc2_sl, z_inc2_sl, r_inc2_sl, speckle_sl, seed_box, smoothing_sl],
                            outputs=[plt_c, plt_rho, plt_r]
                        )

                    # --- Sub-pestaña 2: Transductor ---
                    with gr.TabItem("2. Transductor y pulso RF"):
                        gr.Markdown("### Configuración geométrica del transductor y campo espacial.")
                        with gr.Row():
                            gr.Dropdown(["Nivel 1: continuo + campo por elementos"], value="Nivel 1: continuo + campo por elementos", label="Nivel de simulación")
                            gr.Slider(16, 160, value=136, step=2, label="Número de líneas B-mode")
                            gr.Slider(1, 15, value=5.0, step=0.5, label="Frecuencia central f0 [MHz]")

                    # --- Sub-pestaña 3: Atenuación ---
                    with gr.TabItem("3. Atenuación y K"):
                        gr.Markdown("### Parámetros de atenuación round-trip y factor de dispersión K.")
                        with gr.Row():
                            mu_sk = gr.Slider(0, 2, value=0.5, step=0.1, label="μ [dB/(cm·MHz)]")
                            k_sk = gr.Slider(0.1, 50, value=1.0, step=0.5, label="Factor K")

                    # --- Sub-pestaña 4: Resultado ---
                    with gr.TabItem("4. Resultado: RF y B-mode"):
                        gr.Markdown("### Simulación integral basada en los parámetros configurados.")
                        btn_sim_all = gr.Button("Simular Ultrasonido Completo", variant="primary")
                        with gr.Row():
                            res_bmode_plt = gr.Plot(label="B-mode simulado")
                            res_rf_plt = gr.Plot(label="RF beamformed")

                        def run_full_simulation(cf, rhof, ci1, rhoi1, xi1, zi1, ri1, ci2, rhoi2, xi2, zi2, ri2, sp, sd, sm):
                            phantom_data = ut.generate_custom_phantom(256, 354, 40, 70, cf, rhof, ci1, rhoi1, xi1, zi1, ri1, ci2, rhoi2, xi2, zi2, ri2, sp, sd, sm)
                            params = {"f0": 5.0, "focus": 35.0, "mu": 0.5, "n_lines": 64}
                            res = ut.simulate_ultrasound_engine(phantom_data, params)

                            fig_b, ax_b = plt.subplots(figsize=(5, 6))
                            imb = ax_b.imshow(res["bmode"], extent=[res["x_lines"][0], res["x_lines"][-1], res["z_axis"][-1], res["z_axis"][0]], aspect='auto', cmap='gray', vmin=-60, vmax=0)
                            ax_b.set_title("B-mode Simulado")

                            fig_rf, ax_rf = plt.subplots(figsize=(5, 6))
                            imrf = ax_rf.imshow(res["rf"], extent=[res["x_lines"][0], res["x_lines"][-1], res["z_axis"][-1], res["z_axis"][0]], aspect='auto', cmap='gray')
                            ax_rf.set_title("RF Beamformed")

                            return fig_b, fig_rf

                        btn_sim_all.click(
                            fn=run_full_simulation,
                            inputs=[c_fondo_sl, rho_fondo_sl, c_inc1_sl, rho_inc1_sl, x_inc1_sl, z_inc1_sl, r_inc1_sl, c_inc2_sl, rho_inc2_sl, x_inc2_sl, z_inc2_sl, r_inc2_sl, speckle_sl, seed_box, smoothing_sl],
                            outputs=[res_bmode_plt, res_rf_plt]
                        )

            # --- PESTAÑA PRINCIPAL 2: MODO GUIADO ---
            with gr.TabItem("🎯 Modo Guiado (Misiones)"):
                mission_selector = gr.Dropdown(choices=[("Misión 1", "m1"), ("Misión 2", "m2"), ("Misión 3", "m3")], value="m1", label="Misión Activa")
                theory_md = gr.Markdown(f"**Marco Teórico:** {cm.MISSIONS['m1']['theory']}")
                quiz_radio = gr.Radio(choices=cm.get_shuffled_choices("m1"), label="Hipótesis clínica:")

                with gr.Row():
                    f0_slider = gr.Slider(1.0, 10.0, value=cm.MISSIONS['m1']['default_params']['f0'], label="Frecuencia [MHz]")
                    att_slider = gr.Slider(0.1, 2.0, value=cm.MISSIONS['m1']['default_params']['att'], label="Atenuación")
                    focus_slider = gr.Slider(10.0, 60.0, value=cm.MISSIONS['m1']['default_params']['focus'], label="Foco [mm]")

                btn_sim_m1 = gr.Button("Validar Misión", variant="primary")
                plot_m1 = gr.Plot(label="B-Mode")
                status_box = gr.Textbox(label="Estado", value="Listo.")

                def run_mission_flow(m_id, ans, f0, att, focus):
                    p = sc.generate_phantom()
                    res = sc.simulate_level1_continuous(p, f0_mhz=float(f0), mu_db_cm_mhz=float(att), focus_mm=float(focus))
                    fig = pl.plot_bmode(res)
                    eval_res = cm.validate_mission(m_id, {}, ans)
                    return fig, f"Estado: {eval_res['status']} - {eval_res['feedback']}"

                btn_sim_m1.click(fn=run_mission_flow, inputs=[mission_selector, quiz_radio, f0_slider, att_slider, focus_slider], outputs=[plot_m1, status_box])

            # --- PESTAÑA PRINCIPAL 3: MODO LIBRE ---
            with gr.TabItem("🛠️ Modo Libre (Sandbox)"):
                gr.Markdown("### Exploración libre de parámetros")
                btn_free = gr.Button("Simular Sandbox", variant="primary")
                plt_free = gr.Plot()
                btn_free.click(fn=lambda: pl.plot_bmode(sc.simulate_level1_continuous(sc.generate_phantom())), outputs=[plt_free])

    return demo

if __name__ == "__main__":
    app = build_app()
    app.launch(inline=True, share=True, css=application.LABSTER_CSS)

Overwriting app.py


In [30]:
import importlib
import app as application
importlib.reload(application)

demo = application.build_app()
demo.launch(inline=True, share=True, css=application.LABSTER_CSS)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b5c7cd7cbd084d2294.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [31]:
%%writefile app.py
import gradio as gr
import utilitys as ut
import simulation_core as sc
import plots as pl
import context_manager as cm
import matplotlib.pyplot as plt

LABSTER_CSS = """
:root, .dark, body, .gradio-container {
    background-color: #FFFFFF !important;
    --primary-500: #0284C7 !important;
    --primary-600: #0369A1 !important;
}
.lab-header {
    display: flex; align-items: center; justify-content: space-between;
    padding: 14px 20px; background-color: #F8FAFC; border: 1px solid #E2E8F0;
    border-radius: 6px; margin-bottom: 16px;
}
.brand-title { font-size: 19px; font-weight: 800; color: #0F172A; }
.brand-badge { font-size: 10px; font-weight: 700; background-color: #0284C7; color: #FFFFFF; padding: 2px 6px; border-radius: 3px; }
.module-indicator { color: #64748B; font-size: 13px; font-weight: 500; }
.module-name { color: #0F172A; font-weight: 600; }
"""

def build_app():
    with gr.Blocks(title="MentorIA LAB - US Simulator") as demo:
        gr.HTML("""
        <div class="lab-header">
            <div class="brand-title">Mentor<span style="color: #0284C7;">IA</span> <span class="brand-badge">LAB</span></div>
            <div class="module-indicator">Módulo: <span class="module-name">Laboratorio Completo de Ultrasonido</span></div>
        </div>
        """)

        with gr.Tabs():
            # --- PESTAÑA PRINCIPAL 1: TODO EL ESQUEJO TÉCNICO AGRUPADO ---
            with gr.TabItem("🔬 Laboratorio Técnico (Esqueleto)"):
                with gr.Tabs():
                    # --- Sub-pestaña 1: Phantom ---
                    with gr.TabItem("1. Phantom: c, ρ, Z y R"):
                        gr.Markdown("### Diseño de phantom 2D customizable (Impedancia acústica y microdispersión).")
                        with gr.Row():
                            with gr.Column():
                                nx_sl = gr.Slider(96, 512, value=256, step=16, label="Nx")
                                nz_sl = gr.Slider(64, 640, value=354, step=16, label="Nz")
                                width_sl = gr.Slider(20, 80, value=40, step=5, label="Ancho lateral [mm]")
                                depth_sl = gr.Slider(30, 120, value=70, step=5, label="Profundidad [mm]")
                                c_fondo_sl = gr.Slider(1400, 1650, value=1540, step=10, label="c fondo [m/s]")
                                rho_fondo_sl = gr.Slider(850, 1150, value=1000, step=10, label="ρ fondo [kg/m³]")
                            with gr.Column():
                                gr.Markdown("#### Inclusión 1")
                                c_inc1_sl = gr.Slider(1400, 1700, value=1450, step=10, label="c inc. 1 [m/s]")
                                rho_inc1_sl = gr.Slider(850, 1200, value=950, step=10, label="ρ inc. 1 [kg/m³]")
                                x_inc1_sl = gr.Slider(-30, 30, value=-22, step=1, label="x inc. 1 [mm]")
                                z_inc1_sl = gr.Slider(5, 110, value=19, step=1, label="z inc. 1 [mm]")
                                r_inc1_sl = gr.Slider(1, 20, value=6, step=0.5, label="radio inc. 1 [mm]")
                            with gr.Column():
                                gr.Markdown("#### Inclusión 2")
                                c_inc2_sl = gr.Slider(1400, 1700, value=1650, step=10, label="c inc. 2 [m/s]")
                                rho_inc2_sl = gr.Slider(850, 1200, value=1060, step=10, label="ρ inc. 2 [kg/m³]")
                                x_inc2_sl = gr.Slider(-30, 30, value=9, step=1, label="x inc. 2 [mm]")
                                z_inc2_sl = gr.Slider(5, 110, value=47, step=1, label="z inc. 2 [mm]")
                                r_inc2_sl = gr.Slider(1, 20, value=3.5, step=0.5, label="radio inc. 2 [mm]")
                                speckle_sl = gr.Slider(0, 0.15, value=0.025, step=0.005, label="Microdispersión speckle")
                                seed_box = gr.Number(value=1, label="Seed")
                                smoothing_sl = gr.Slider(0, 3, value=0.7, step=0.1, label="Suavizado interfaces [pix]")

                        btn_gen_phantom = gr.Button("Generar Phantom", variant="primary")
                        with gr.Row():
                            plt_c = gr.Plot(label="Velocidad c(x,z)")
                            plt_rho = gr.Plot(label="Densidad ρ(x,z)")
                            plt_r = gr.Plot(label="Reflectividad |R(x,z)|")

                        def update_phantom(nx, nz, w, d, cf, rhof, ci1, rhoi1, xi1, zi1, ri1, ci2, rhoi2, xi2, zi2, ri2, sp, sd, sm):
                            data = ut.generate_custom_phantom(int(nx), int(nz), w, d, cf, rhof, ci1, rhoi1, xi1, zi1, ri1, ci2, rhoi2, xi2, zi2, ri2, sp, sd, sm)

                            fig1, ax1 = plt.subplots(figsize=(4, 4))
                            im1 = ax1.imshow(data["c_map"], extent=[data["x_axis"][0], data["x_axis"][-1], data["z_axis"][-1], data["z_axis"][0]], aspect='auto', cmap='viridis')
                            fig1.colorbar(im1, ax=ax1, fraction=0.046)

                            fig2, ax2 = plt.subplots(figsize=(4, 4))
                            im2 = ax2.imshow(data["rho_map"], extent=[data["x_axis"][0], data["x_axis"][-1], data["z_axis"][-1], data["z_axis"][0]], aspect='auto', cmap='plasma')
                            fig2.colorbar(im2, ax=ax2, fraction=0.046)

                            fig3, ax3 = plt.subplots(figsize=(4, 4))
                            im3 = ax3.imshow(data["reflectivity"], extent=[data["x_axis"][0], data["x_axis"][-1], data["z_axis"][-1], data["z_axis"][0]], aspect='auto', cmap='gray')
                            fig3.colorbar(im3, ax=ax3, fraction=0.046)

                            return fig1, fig2, fig3

                        btn_gen_phantom.click(
                            fn=update_phantom,
                            inputs=[nx_sl, nz_sl, width_sl, depth_sl, c_fondo_sl, rho_fondo_sl, c_inc1_sl, rho_inc1_sl, x_inc1_sl, z_inc1_sl, r_inc1_sl, c_inc2_sl, rho_inc2_sl, x_inc2_sl, z_inc2_sl, r_inc2_sl, speckle_sl, seed_box, smoothing_sl],
                            outputs=[plt_c, plt_rho, plt_r]
                        )

                    # --- Sub-pestaña 2: Transductor ---
                    with gr.TabItem("2. Transductor y pulso RF"):
                        gr.Markdown("### Configuración geométrica del transductor y campo espacial.")
                        with gr.Row():
                            gr.Dropdown(["Nivel 1: continuo + campo por elementos"], value="Nivel 1: continuo + campo por elementos", label="Nivel de simulación")
                            gr.Slider(16, 160, value=136, step=2, label="Número de líneas B-mode")
                            gr.Slider(1, 15, value=5.0, step=0.5, label="Frecuencia central f0 [MHz]")

                    # --- Sub-pestaña 3: Atenuación ---
                    with gr.TabItem("3. Atenuación y K"):
                        gr.Markdown("### Parámetros de atenuación round-trip y factor de dispersión K.")
                        with gr.Row():
                            mu_sk = gr.Slider(0, 2, value=0.5, step=0.1, label="μ [dB/(cm·MHz)]")
                            k_sk = gr.Slider(0.1, 50, value=1.0, step=0.5, label="Factor K")

                    # --- Sub-pestaña 4: Resultado ---
                    with gr.TabItem("4. Resultado: RF y B-mode"):
                        gr.Markdown("### Simulación integral basada en los parámetros configurados.")
                        btn_sim_all = gr.Button("Simular Ultrasonido Completo", variant="primary")
                        with gr.Row():
                            res_bmode_plt = gr.Plot(label="B-mode simulado")
                            res_rf_plt = gr.Plot(label="RF beamformed")

                        def run_full_simulation(cf, rhof, ci1, rhoi1, xi1, zi1, ri1, ci2, rhoi2, xi2, zi2, ri2, sp, sd, sm):
                            phantom_data = ut.generate_custom_phantom(256, 354, 40, 70, cf, rhof, ci1, rhoi1, xi1, zi1, ri1, ci2, rhoi2, xi2, zi2, ri2, sp, sd, sm)
                            params = {"f0": 5.0, "focus": 35.0, "mu": 0.5, "n_lines": 64}
                            res = ut.simulate_ultrasound_engine(phantom_data, params)

                            fig_b, ax_b = plt.subplots(figsize=(5, 6))
                            imb = ax_b.imshow(res["bmode"], extent=[res["x_lines"][0], res["x_lines"][-1], res["z_axis"][-1], res["z_axis"][0]], aspect='auto', cmap='gray', vmin=-60, vmax=0)
                            ax_b.set_title("B-mode Simulado")

                            fig_rf, ax_rf = plt.subplots(figsize=(5, 6))
                            imrf = ax_rf.imshow(res["rf"], extent=[res["x_lines"][0], res["x_lines"][-1], res["z_axis"][-1], res["z_axis"][0]], aspect='auto', cmap='gray')
                            ax_rf.set_title("RF Beamformed")

                            return fig_b, fig_rf

                        btn_sim_all.click(
                            fn=run_full_simulation,
                            inputs=[c_fondo_sl, rho_fondo_sl, c_inc1_sl, rho_inc1_sl, x_inc1_sl, z_inc1_sl, r_inc1_sl, c_inc2_sl, rho_inc2_sl, x_inc2_sl, z_inc2_sl, r_inc2_sl, speckle_sl, seed_box, smoothing_sl],
                            outputs=[res_bmode_plt, res_rf_plt]
                        )

            # --- PESTAÑA PRINCIPAL 2: MODO GUIADO ---
            with gr.TabItem("🎯 Modo Guiado (Misiones)"):
                mission_selector = gr.Dropdown(choices=[("Misión 1", "m1"), ("Misión 2", "m2"), ("Misión 3", "m3")], value="m1", label="Misión Activa")
                theory_md = gr.Markdown(f"**Marco Teórico:** {cm.MISSIONS['m1']['theory']}")
                quiz_radio = gr.Radio(choices=cm.get_shuffled_choices("m1"), label="Hipótesis clínica:")

                with gr.Row():
                    f0_slider = gr.Slider(1.0, 10.0, value=cm.MISSIONS['m1']['default_params']['f0'], label="Frecuencia [MHz]")
                    att_slider = gr.Slider(0.1, 2.0, value=cm.MISSIONS['m1']['default_params']['att'], label="Atenuación")
                    focus_slider = gr.Slider(10.0, 60.0, value=cm.MISSIONS['m1']['default_params']['focus'], label="Foco [mm]")

                btn_sim_m1 = gr.Button("Validar Misión", variant="primary")
                plot_m1 = gr.Plot(label="B-Mode")
                status_box = gr.Textbox(label="Estado", value="Listo.")

                def run_mission_flow(m_id, ans, f0, att, focus):
                    p = sc.generate_phantom()
                    res = sc.simulate_level1_continuous(p, f0_mhz=float(f0), mu_db_cm_mhz=float(att), focus_mm=float(focus))
                    fig = pl.plot_bmode(res)
                    eval_res = cm.validate_mission(m_id, {}, ans)
                    return fig, f"Estado: {eval_res['status']} - {eval_res['feedback']}"

                btn_sim_m1.click(fn=run_mission_flow, inputs=[mission_selector, quiz_radio, f0_slider, att_slider, focus_slider], outputs=[plot_m1, status_box])

            # --- PESTAÑA PRINCIPAL 3: MODO LIBRE ---
            with gr.TabItem("🛠️ Modo Libre (Sandbox)"):
                gr.Markdown("### Exploración libre de parámetros")
                btn_free = gr.Button("Simular Sandbox", variant="primary")
                plt_free = gr.Plot()
                btn_free.click(fn=lambda: pl.plot_bmode(sc.simulate_level1_continuous(sc.generate_phantom())), outputs=[plt_free])

    return demo

if __name__ == "__main__":
    app = build_app()
    app.launch(inline=True, share=True, css=application.LABSTER_CSS)

Overwriting app.py


In [32]:
import importlib
import app as application
importlib.reload(application)

demo = application.build_app()
demo.launch(inline=True, share=True, css=application.LABSTER_CSS)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7a09b055397c75f48c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
